# Nhận diện trạng thái ô đỗ xe — PKLot


## 1. Bài toán và mục tiêu


### Bối cảnh và mục tiêu

Trong bãi xe ngoài trời, việc xác định vị trí còn trống từ camera có thể hỗ trợ người quản lý theo dõi mức sử dụng bãi mà không cần kiểm tra từng ô trực tiếp. Bài toán gặp khó khăn khi xe ở xa camera, bóng đổ che đường kẻ, ánh sáng thay đổi hoặc mặt đường bị ảnh hưởng bởi mưa.

Đề tài xây dựng hệ thống nhận diện trạng thái ô đỗ và đếm số chỗ trống từ ảnh màu toàn cảnh. Hệ thống được thiết kế để so sánh ba phương pháp: phân loại ảnh từng ô bằng HOG kết hợp SVM, phát hiện xe bằng YOLO11n trên toàn ảnh và phát hiện xe bằng cùng detector trên các ROI/tile. Việc so sánh xem xét chất lượng nhận diện, sai số đếm và chi phí xử lý; lợi ích của chia tile cần được kiểm chứng bằng số liệu.

### Đầu vào và đầu ra

Đầu vào gồm ảnh màu toàn cảnh từ camera cố định và cấu hình các ô đỗ tương ứng với ảnh. Mỗi ô có mã định danh và polygon biểu diễn vùng ô trong hệ tọa độ ảnh gốc. Ngày thu nhận, camera và thời tiết được giữ lại để chia dữ liệu và phân tích kết quả. Cấu hình phương pháp gồm checkpoint, kích thước đầu vào, các quy tắc ghép xe với ô và ngưỡng đã lựa chọn trên validation.

Đầu ra của mỗi ảnh gồm mã ô, vị trí vùng ô, trạng thái có xe/trống/chưa xác định và điểm có xe khi phương pháp cung cấp được. Tổng số chỗ trống được tính từ những ô được xác định là trống; số ô chưa xác định được báo riêng. Kết quả được trực quan hóa trên ảnh gốc và lưu dưới dạng CSV/JSON để đánh giá và đối chiếu.

### Phân biệt phát hiện xe và nhận diện trạng thái ô

Phát hiện xe trả về vị trí hộp bao của các đối tượng xe và độ tin cậy. Nhận diện trạng thái ô trả lời một vùng ô đỗ đã biết đang có xe hay trống. Với phương pháp dựa trên detector, cần thêm bước ánh xạ hộp xe về ảnh gốc và ghép hình học xe–ô trước khi suy ra trạng thái. Với HOG/SVM, ảnh của từng ô được phân loại trực tiếp theo nhãn trạng thái.

Polygon hoặc hình chữ nhật xoay trong XML PKLot biểu diễn vùng ô đỗ; chúng không xác định hộp bao của chiếc xe. Vì vậy, nhãn ô không được chuyển thành nhãn hộp xe để huấn luyện detector hoặc tính mAP phát hiện xe. Phần đánh giá chính sử dụng trạng thái từng ô, phép đếm và thời gian trên ảnh gốc.

### Phạm vi và giả định

Phạm vi chính là bãi xe ngoài trời với camera cố định và các vùng ô đỗ đã được cung cấp. Khi vị trí hoặc góc nhìn camera thay đổi, cấu hình vùng ô phải được cập nhật hoặc xác nhận lại. Hệ thống không giả định một cấu hình polygon duy nhất cho mọi ảnh nếu dữ liệu ghi nhận nhiều biến thể hình học.

Canny và Hough được dùng trên nhánh ảnh xám để phân tích biên, đường kẻ và kiểm tra chất lượng hình học. Nhánh detector sử dụng ảnh màu; việc không tìm thấy đường kẻ không được dùng riêng để kết luận ô trống. Tự động đề xuất vị trí ô đỗ là hướng mở rộng, chưa thuộc phạm vi thực nghiệm chính. Kết quả sẽ được phân tích theo thời tiết và camera; thí nghiệm chuyển góc nhìn trong cùng bãi được trình bày riêng.


## 2. Môi trường và cấu hình


In [ ]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from urllib.parse import quote


In [ ]:
OWNER = "tuan314159265"
REPO = "parking-occupancy"
REF = "main"  # Nhánh/tag/commit SHA. Chạy chính thức: thay bằng SHA commit chứa parking_module.py để tái lập chính xác.
MODULE_PATH = "parking_module.py"
 
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = "parking-occupancy"  # Thư mục CÁ NHÂN để lưu kết quả/module.
DOWNLOAD_DATA = True
FORCE_AUDIT = False  # True nếu dataset đã thay đổi hoặc muốn kiểm tra lại toàn bộ.

# Để trống: dùng data/raw/PKLot trong WORK_DIR; dữ liệu đã có sẽ được dùng lại.
# Dùng chung: thêm shortcut Drive, rồi điền thư mục chứa camera UFPR04/UFPR05/PUCPR.
SHARED_DATASET_DIR = ""  # Local: dùng data/raw/PKLot trong repo. Chỉ điền khi dùng dataset ở vị trí khác.
# Không bắt buộc: thư mục outputs/full của chủ dataset (chỉ đọc).
SHARED_AUDIT_DIR = ""  # Ví dụ /content/drive/MyDrive/PKLot-shared/outputs/full

# Bộ manifest dùng chung, đã khóa trên GitHub.
PROJECT_DATASET_VERSION = "pklot_v1"
PROTOCOL = "within_camera"  # Hoặc transfer_ufpr04_to_ufpr05; phải huấn luyện lại trên split của giao thức đó.

# Runner chung cho validation VÀ test; tên cờ giữ tương thích với các cell demo cũ.
VALIDATION_ONLY_MODE = True
RUN_FINAL_TEST = False  # Chỉ bật sau khi xem và xác nhận locked_config.json của task 12.
TASK12_RUN_NAME = "validation_v1"  # Đổi tên nếu plan/code/môi trường thay đổi.

# Luồng một notebook: có khóa rồi thì bỏ tìm kiếm validation, chạy tiếp mục 6.3.
RUN_VALIDATION_SEARCH = False  # Chỉ True khi cần làm validation mới theo plan đã chốt.
PREFER_LOCAL_FILES = True  # Dùng code/split/plan trong repo hoặc bundle trước khi tải GitHub.
LOCKED_BUNDLE_ZIP = ""  # Colab mới: đường dẫn parking-final-test-bundle.zip đã đưa lên Drive/runtime.
VALIDATION_ARTIFACT_DIR = ""  # Để trống để tìm model/khóa đã có; hoặc điền thư mục validation_v1.
FINAL_TEST_OUTPUT_DIR = ""  # Để trống: OUTPUT_DIR/PROTOCOL/locked_test_v1.
FINAL_TEST_RUN_NAME = "locked_test_v1"  # Đổi tên khi cần một lượt mới, không ghi đè kết quả.
RUN_DEVICE = "auto"  # Local dùng CPU khi không có CUDA; ghi đúng thiết bị khi báo timing.
TEST_EXAMPLES_PER_CATEGORY = 1


### Kết nối Google Drive

Trong cửa sổ cấp quyền Google, chọn tài khoản Drive của bạn, đánh dấu **tất cả các quyền được yêu cầu** (hoặc **Select all**) rồi bấm **Continue**. Bỏ sót một quyền có thể gây lỗi `credential propagation was unsuccessful`. Không chỉ chọn riêng quyền đọc/ghi Drive.

Nếu không thấy cửa sổ, kiểm tra popup bị chặn. Nếu vẫn lỗi dù đã cấp đủ quyền, thử Runtime → Disconnect and delete runtime rồi kết nối lại. Thao tác này xóa dữ liệu tạm của runtime, giữ file trên Drive. Nếu dùng local runtime, đặt USE_GOOGLE_DRIVE=False.

Nguồn: https://github.com/googlecolab/colabtools/issues/6089


In [ ]:
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        from google.colab._message import MessageError
    except ImportError as exc:
        raise RuntimeError(
            "Kết nối bằng runtime Google Colab để dùng Drive. "
            "Nếu chạy local, đặt USE_GOOGLE_DRIVE=False."
        ) from exc

    mountpoint = Path("/content/drive")
    def drive_ready():
        return os.path.ismount(mountpoint) and (mountpoint / "MyDrive").is_dir()

    if not drive_ready():
        for attempt in range(2):
            try:
                print("Kết nối Google Drive: chọn tài khoản, chọn TẤT CẢ quyền được yêu cầu rồi Continue.")
                drive.mount(str(mountpoint), force_remount=(attempt == 1), timeout_ms=120000)
                if not drive_ready():
                    raise RuntimeError("Drive chưa có thư mục MyDrive sau khi mount.")
                break
            except MessageError as exc:
                raise RuntimeError(
                    "Colab chưa nhận được thông tin xác thực Google (MessageError). "
                    "Chạy lại cell này; trong hộp thoại đăng nhập, chọn đúng tài khoản "
                    "và đánh dấu tất cả quyền Drive/Colab được yêu cầu trước khi Continue. "
                    "Nếu không thấy hộp thoại, kiểm tra trình duyệt có chặn popup. "
                    "Nếu vẫn lỗi, dùng Runtime > Disconnect and delete runtime rồi kết nối lại. "
                    "Chưa tải dataset và chưa chuyển sang bộ nhớ tạm. "
                    f"Chi tiết gốc: {exc}"
                ) from exc
            except (ValueError, OSError, RuntimeError) as exc:
                if attempt == 0:
                    print("Kết nối chưa thành công; thử lại bằng force_remount=True.")
                else:
                    raise RuntimeError(
                        "Không kết nối được Google Drive sau 2 lần thử. "
                        "Nếu dùng local runtime, chuyển sang runtime Google Colab. "
                        "Trên Colab: chọn Runtime > Disconnect and delete runtime, "
                        "kết nối lại rồi Run All và cấp quyền Drive. "
                        "Để chạy tạm không dùng Drive, đặt USE_GOOGLE_DRIVE=False; "
                        "khi đó dữ liệu chỉ nằm trong phiên Colab. "
                        "Chưa tải dataset vì Drive chưa sẵn sàng."
                    ) from exc
    WORK_DIR = mountpoint / "MyDrive" / DRIVE_PROJECT_DIR
    print("Dữ liệu và kết quả sẽ được lưu trên Google Drive.")
else:
    if Path("/content").is_dir():
        WORK_DIR = Path("/content/parking-occupancy")
        print("Chế độ không dùng Drive. Dữ liệu Colab chỉ lưu trong runtime.")
    else:
        # Local kernels may start in notebooks/ instead of the repository root.
        current_dir = Path.cwd().resolve()
        WORK_DIR = next(
            (directory for directory in (current_dir, *current_dir.parents)
             if (directory / "configs/pklot_v1.json").is_file()
             and (directory / "parking_module.py").is_file()),
            current_dir,
        )
        print("Chế độ local. Dùng lại dữ liệu trong thư mục dự án.")

RAW_DIR = WORK_DIR / "data/raw"
DATASET_DIR = Path(SHARED_DATASET_DIR) if SHARED_DATASET_DIR else RAW_DIR / "PKLot"
DRIVE_OUTPUT_DIR = WORK_DIR / "outputs/full"
if USE_GOOGLE_DRIVE:
    # Drive cho phép nhiều thư mục trùng tên; mkdir lặp lại khi thư mục chưa hiện kịp sẽ tạo thêm bản sao.
    # Vì vậy mọi kết quả được ghi vào đĩa cục bộ của Colab, rồi sync_outputs_to_drive() chép lên Drive.
    OUTPUT_DIR = Path("/content/parking-occupancy-local/outputs/full")
else:
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
WORK_DIR.mkdir(parents=True, exist_ok=True)

import shutil

_DRIVE_DIRS_CREATED = set()


def _wait_until(predicate, timeout=60.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if predicate():
            return True
        time.sleep(0.5)
    return False


def _ensure_drive_dir(path):
    """Tạo thư mục trên Drive đúng một lần rồi chờ nó hiện ra; không bao giờ mkdir lại (tránh thư mục trùng tên)."""
    if path.is_dir():
        _DRIVE_DIRS_CREATED.add(path)
        return
    if path not in _DRIVE_DIRS_CREATED:
        _ensure_drive_dir(path.parent)
        try:
            path.mkdir()
        except FileExistsError:
            pass
        _DRIVE_DIRS_CREATED.add(path)
    if not _wait_until(path.is_dir):
        raise RuntimeError(f"Thư mục Drive chưa hiện ra sau 60s: {path}. Chạy lại cell này để đồng bộ tiếp.")


def sync_outputs_to_drive():
    """Chép kết quả mới hoặc đã đổi từ đĩa cục bộ lên Drive. Chạy lại bao nhiêu lần cũng an toàn."""
    if OUTPUT_DIR == DRIVE_OUTPUT_DIR:
        return
    copied = 0
    for source in sorted(OUTPUT_DIR.rglob("*")):
        # Feature matrices/cache lớn chỉ giữ local; model và báo cáo vẫn được lưu Drive.
        relative = source.relative_to(OUTPUT_DIR)
        if any(part in ("features", "detections") for part in relative.parts) or source.suffix in (".dat", ".npy"):
            continue
        target = DRIVE_OUTPUT_DIR / source.relative_to(OUTPUT_DIR)
        if source.is_dir():
            _ensure_drive_dir(target)
            continue
        _ensure_drive_dir(target.parent)
        if target.exists() and target.stat().st_size == source.stat().st_size                 and target.stat().st_mtime >= source.stat().st_mtime:
            continue
        for attempt in range(5):
            try:
                shutil.copyfile(source, target)
                break
            except OSError as error:
                if attempt == 4:
                    raise
                print(f"  ⚠ Chép {source.name} lỗi ({error.__class__.__name__}); thử lại {attempt + 1}/4")
                time.sleep(1.5 * (attempt + 1))
        copied += 1
    print(f"Đồng bộ Drive: {copied} file mới/đã đổi -> {DRIVE_OUTPUT_DIR}")


def restore_outputs_from_drive():
    """Kéo kết quả của các lần chạy trước từ Drive về đĩa cục bộ (chỉ đọc Drive)."""
    if OUTPUT_DIR == DRIVE_OUTPUT_DIR or not DRIVE_OUTPUT_DIR.is_dir():
        return
    restored = 0
    for source in sorted(DRIVE_OUTPUT_DIR.rglob("*")):
        relative = source.relative_to(DRIVE_OUTPUT_DIR)
        if any(part in ("features", "detections") for part in relative.parts) or source.suffix in (".dat", ".npy"):
            continue
        target = OUTPUT_DIR / source.relative_to(DRIVE_OUTPUT_DIR)
        if source.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        elif not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
            restored += 1
    print(f"Khôi phục từ Drive: {restored} file.")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
restore_outputs_from_drive()
print("Thư mục lưu:", WORK_DIR)


### Dùng lại model và khóa validation trên cùng notebook

Local đã có artifact thì để `LOCKED_BUNDLE_ZIP=""`. Với runtime Colab mới, đưa ZIP đã chuẩn bị lên Drive/runtime và điền đường dẫn ở mục 2. Cell này chỉ giải nén/kiểm tra gói, không chạy model và không chứa ảnh PKLot. Dataset vẫn dùng `DATASET_DIR` riêng.

Thư viện phải khớp khóa. Nếu runtime không hỗ trợ các phiên bản đã dùng, chạy local với `.venv-validation`; không sửa hash để bỏ qua kiểm tra.


In [ ]:
# Khôi phục model/khóa/code cùng notebook trên Colab; không cần khi chạy repo local.
from zipfile import ZipFile
LOCKED_BUNDLE_ROOT = WORK_DIR if (WORK_DIR / "bundle_manifest.json").is_file() else WORK_DIR / "outputs/locked_bundle"
if LOCKED_BUNDLE_ZIP:
    archive_path = Path(LOCKED_BUNDLE_ZIP).expanduser()
    if not archive_path.is_file():
        raise FileNotFoundError(f"Không có ZIP: {archive_path}")
    target = LOCKED_BUNDLE_ROOT.resolve()
    if not (target / "bundle_manifest.json").is_file():
        if target.exists() and any(target.iterdir()):
            raise FileExistsError(f"Không giải nén đè thư mục có dữ liệu: {target}")
        with ZipFile(archive_path) as archive:
            for member in archive.infolist():
                if not (target / member.filename).resolve().is_relative_to(target):
                    raise ValueError("Đường dẫn ZIP không hợp lệ")
            archive.extractall(target)
    bundle_manifest = json.loads((target / "bundle_manifest.json").read_text())
    for name, expected in bundle_manifest["files"].items():
        artifact = (target / name).resolve()
        if not artifact.is_relative_to(target):
            raise ValueError("Đường dẫn manifest bundle không hợp lệ")
        if hashlib.sha256(artifact.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Artifact bundle sai checksum: {name}")
    print("Đã kiểm tra bundle:", target)
else:
    print("Không yêu cầu giải nén bundle; dùng file local/Drive đã có.")


In [ ]:
# Thư viện dùng chung cho các phần xử lý và trực quan hóa.
required_packages = {
    "PIL": "Pillow>=10,<13",
    "cv2": "opencv-python-headless>=4.10,<5",
    "numpy": "numpy>=1.26,<3",
    "matplotlib": "matplotlib>=3.9,<4",
    "ultralytics": "ultralytics==8.4.0",
    "skimage": "scikit-image>=0.25,<1",
    "sklearn": "scikit-learn>=1.6,<2",
    "joblib": "joblib>=1.4,<2",
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
import importlib.metadata
if importlib.util.find_spec("ultralytics") is not None and importlib.metadata.version("ultralytics") != "8.4.0":
    missing_packages.append("ultralytics==8.4.0")
if missing_packages:
    missing_packages.append("opencv-python>=4.10,<5")
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)

import cv2
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

print("Library versions:", {"opencv": cv2.__version__, "numpy": np.__version__,
                            "matplotlib": mpl.__version__})


In [ ]:
assert OWNER != "YOUR_GITHUB_USERNAME", "Điền OWNER, REPO và REF ở cell cấu hình"
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODULE_CACHE_DIR = WORK_DIR / "outputs/runtime"
MODULE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
local_candidates = [Path(globals().get("LOCKED_BUNDLE_ROOT", WORK_DIR / "outputs/locked_bundle")) / MODULE_PATH,
                    WORK_DIR / MODULE_PATH]
local_source = next((path for path in local_candidates if path.is_file()), None) if globals().get("PREFER_LOCAL_FILES", False) else None
url = None
if local_source is not None:
    module_file = local_source
    code = module_file.read_bytes()
    print("Dùng module đã có:", module_file)
else:
    url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/"
           f"{quote(REPO, safe='')}/{quote(REF, safe='')}/{quote(MODULE_PATH, safe='/')}")
    request = urllib.request.Request(f"{url}?refresh={time.time_ns()}", headers={"Cache-Control": "no-cache"})
    with urllib.request.urlopen(request, timeout=60) as response:
        code = response.read()
    module_file = MODULE_CACHE_DIR / "parking_module.py"
    module_file.write_bytes(code)
compile(code, MODULE_PATH, "exec")
spec = importlib.util.spec_from_file_location("parking_module", module_file)
parking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parking
spec.loader.exec_module(parking)
provenance = {"url": url, "local_file": str(module_file), "ref": REF, "sha256": hashlib.sha256(code).hexdigest()}
(MODULE_CACHE_DIR / "module_source.json").write_text(json.dumps(provenance, indent=2))
print("Loaded module:", provenance)


### Giao diện dùng chung trong một notebook

`parking_module.py` chứa loader, hình học, ROI/tile, matching, evaluator và runner validation/test đã khóa. `final_test_module.py` kiểm tra đầu vào, gọi runner và xuất bảng/ảnh/checksum từ dự đoán đã lưu; helper được lấy từ repo, bundle hoặc cùng REF trên GitHub.

- Dữ liệu/đánh giá: `read_project_frame`, `evaluate_occupancy`, `FrameTimer`, `export_evaluation`.
- Hình học: `analyze_parking_geometry`, `build_geometry_reference`, `assess_geometry_quality`, `apply_geometry_policy`.
- ROI/matching: `make_parking_tiles`, `tile_boxes_to_image`, `infer_parking_occupancy`.
- Thí nghiệm: `run_validation_search`, `verify_validation_lock`, `run_locked_test`.
- Bằng chứng: `final_test.check_inputs`, `run_final_test`, `export_tables`, `export_examples`, `seal_evidence`.

Các demo mục 4–5 là tùy chọn; luồng chính ở 6.2–6.3 tự chạy thành phần cần thiết. Polygon XML là vùng ô, không phải box xe.


## 3. Dữ liệu và tiền xử lý


### 3.1. Tải và kiểm tra dữ liệu PKLot


#### Nguồn dữ liệu và tổ chức PKLot

PKLot được lựa chọn vì cung cấp ảnh bãi xe toàn cảnh cùng thông tin vùng ô và trạng thái chiếm dụng. Theo công bố của bộ dữ liệu, PKLot có 12.417 ảnh màu độ phân giải 1280 × 720 pixel, thu nhận tại hai bãi xe ngoài trời trong các điều kiện nắng, nhiều mây và mưa. Nguồn dữ liệu và công trình giới thiệu PKLot được dẫn tại [trang PKLot của UFPR](https://web.inf.ufpr.br/vri/databases/parking-lot-database/) và [tài liệu giới thiệu của tác giả](https://www.inf.ufpr.br/lesoliveira/download/pklot-readme.pdf).

Archive sử dụng trong dự án có ba góc nhìn PUCPR, UFPR04 và UFPR05. UFPR04 và UFPR05 quan sát cùng bãi UFPR từ hai góc nhìn; PUCPR thuộc bãi khác. Ảnh được tổ chức theo camera, thời tiết và ngày. Dự án sử dụng ảnh toàn cảnh và XML để giữ bối cảnh cho detector; bộ ảnh từng ô đã cắt sẵn không thay thế đầu vào toàn cảnh.

#### Nhãn ô đỗ và cấu hình hình học

Mỗi phần tử space trong XML có mã ô, nhãn occupied và contour của vùng ô. Nhãn 1 biểu thị có xe, nhãn 0 biểu thị trống. Nhãn thiếu hoặc không hợp lệ không được suy đoán thành nhãn trống. Trạng thái chưa xác định là đầu ra mà hệ thống có thể sử dụng khi chưa đủ cơ sở quyết định; đó không phải lớp ground truth bổ sung của tập con đã chốt.

Tọa độ polygon được giữ trong ảnh gốc, gốc ở góc trên trái, trục x hướng sang phải và trục y hướng xuống dưới. Bộ JSON hình học của tập con có 5 layout: PUCPR có 1, UFPR04 có 3 và UFPR05 có 1. Mỗi frame liên kết đến layout bằng geometry_id, còn nhãn trạng thái được lưu riêng theo mã ô. Cách biểu diễn này giữ được các biến thể chú thích thay vì ép mọi ảnh cùng camera dùng một polygon cố định.

#### Kiểm tra chất lượng và bảo toàn metadata

Với archive local đã kiểm tra bằng loader v2, dự án ghi nhận 12.416 XML và 1 ảnh thiếu XML. Trong số các frame có XML, 11.638 frame hợp lệ được giữ lại; 778 frame bị loại do nhãn hoặc mã ô không hợp lệ. Số lượng này là kết quả kiểm tra của dự án, khác với số ảnh được công bố cho toàn bộ PKLot. Danh sách lỗi được lưu để truy vết; không tự gán nhãn thay thế hoặc bỏ âm thầm ô lỗi trong một frame.

Loader hỗ trợ cả hai cách viết Point/point trong contour. Ngày thu nhận chuẩn được lấy từ timestamp hợp lệ trong tên file; ngày thư mục gốc được giữ trong directory_date và nguồn ngày được ghi trong date_source. Có 195 frame hợp lệ có ngày trong tên file khác ngày thư mục, bao gồm thư mục có tháng không hợp lệ. Dữ liệu gốc không bị đổi tên. Thời tiết Cloudy được chuẩn hóa thành overcast, đồng thời đường dẫn gốc vẫn được lưu.

Bảng dưới cho biết số frame hợp lệ trước khi chọn tập con thực nghiệm.

| Camera | Bãi xe | Frame hợp lệ |
| --- | --- | --- |
| PUCPR | PUCPR | 4.077 |
| UFPR04 | UFPR | 3.547 |
| UFPR05 | UFPR | 4.014 |
| Tổng |  | 11.638 |


#### Dùng lại dataset đã có

Có dataset XML: bỏ qua tải/giải nén, kể cả thiếu provenance.json. Có audit tương thích: dùng lại manifest và kiểm tra nhanh một mẫu, bỏ qua audit toàn bộ. Thiếu cache: audit một lần và lưu vào thư mục cá nhân. FORCE_AUDIT=True để quét lại sau khi dữ liệu thay đổi. Cache không chứng minh dataset chưa thay đổi; chỉ dùng lại khi bộ dữ liệu được giữ nguyên.

Với dữ liệu chung, điền SHARED_DATASET_DIR và tùy chọn SHARED_AUDIT_DIR. Mã chỉ đọc các thư mục này; không tải hay giải nén vào đó. Nếu đường dẫn chung sai thì dừng để bạn sửa. DRIVE_PROJECT_DIR phải là thư mục cá nhân; không trỏ nó vào shortcut chung. Archive khoảng 4,9 GB; lần đầu cần khoảng 10–15 GB trống nếu tự tải.


In [ ]:
summary = parking.ensure_dataset(
    raw=RAW_DIR,
    output=OUTPUT_DIR,
    dataset_root=DATASET_DIR if SHARED_DATASET_DIR else None,
    cached_audit_dir=SHARED_AUDIT_DIR or None,
    download=DOWNLOAD_DATA,
    force_audit=FORCE_AUDIT,
)
print(json.dumps(summary, indent=2))
assert summary["valid_frames"] > 0


In [ ]:
# Đọc một mẫu hợp lệ trực tiếp bằng module lấy từ GitHub.
with (OUTPUT_DIR / "valid_samples.jsonl").open() as stream:
    sample = json.loads(next(stream))
frame = parking.read_frame(DATASET_DIR / sample["xml_path"], DATASET_DIR)
print("Sample:", frame.sample_id)
print("Camera / weather / date:", frame.camera, frame.weather, frame.date)
print("Parking spaces:", len(frame.spaces))
print("Outputs:", OUTPUT_DIR)

globals().get("sync_outputs_to_drive", lambda: None)()

### 3.2. Chia tập dữ liệu và cấu hình vùng ô đỗ

#### Tập con dùng chung và ngân sách xử lý

Để phù hợp với ngân sách Colab và giữ so sánh công bằng, dự án cố định tập con pklot_v1 dùng chung cho mọi phương pháp. Seed là 42; mỗi cặp camera/thời tiết có giới hạn tối đa 140 frame train, 30 validation và 30 test. Việc chọn frame luân phiên theo ngày và dùng thứ tự hash xác định; không lặp ảnh để đạt số lượng mong muốn.

Giao thức chính có tối đa 1.800 ảnh và thực tế chọn được 1.722 ảnh. Hai nhóm train UFPR04/rainy và UFPR05/rainy thiếu lần lượt 22 và 56 ảnh so với giới hạn; không lấy bù từ validation/test. Tập con ưu tiên khả năng so sánh giữa camera/thời tiết, nên phân bố này không đại diện tỉ lệ thời tiết tự nhiên của toàn bộ PKLot.

#### Chia theo nhóm ngày và kiểm thử chuyển camera

Train, validation và test được chia theo ngày thu nhận, không chia ngẫu nhiên từng frame hoặc từng crop ô. Một ngày chỉ thuộc một split trong mỗi giao thức, kể cả khi ngày đó có nhiều thời tiết hoặc camera; tất cả ô của một frame đi cùng split. Quy tắc này hạn chế việc ảnh gần nhau theo thời gian, chứa cùng xe đỗ, xuất hiện đồng thời trong tập học và tập kiểm thử.

Giao thức within_camera đưa cả ba camera vào mỗi split và phân chia nhóm ngày xấp xỉ 70/15/15. Phương án chia được chọn trong 256 ứng viên xác định bởi seed, dựa vào số lượng và metadata camera/thời tiết, không dựa vào dự đoán hoặc điểm số test. Sau khi áp dụng giới hạn ảnh, số frame thực tế được trình bày trong bảng; tỉ lệ nhóm ngày không được hiểu là tỉ lệ ảnh chính xác.

Giao thức transfer_ufpr04_to_ufpr05 dùng UFPR04 cho train/validation và giữ UFPR05 cho test. Mọi ngày của camera đích được loại khỏi nguồn trước khi chia ngày nguồn xấp xỉ 80/20. Đây là kiểm thử góc nhìn mới trong cùng bãi UFPR, không phải kiểm thử chuyển sang một bãi xe độc lập. Hai giao thức là hai thí nghiệm riêng, không gộp manifest hoặc dùng model đã học camera đích để báo kết quả chuyển camera.

#### Vai trò của các split và khả năng tái lập

Train được sử dụng cho các thành phần cần học, như HOG/SVM. Validation được dùng để chọn tham số, padding, overlap và ngưỡng suy trạng thái. Với detector pretrained, checkpoint và cấu hình được chốt trước khi đánh giá test. Sau khi khóa lựa chọn, tất cả phương pháp chạy trên cùng tập test của giao thức tương ứng; không chọn lại tham số bằng kết quả test.

Các manifest train/val/test lưu đường dẫn tương đối, camera, bãi xe, thời tiết, ngày, mã layout và nhãn từng ô. File split_lock.json ghi cấu hình, danh sách ngày và SHA256 các artifact; notebook tải cùng bộ metadata từ GitHub thay vì tự chia lại trên mỗi máy. Bộ kiểm tra xác nhận không giao ngày giữa các split và chỉ có camera đích trong test của thí nghiệm chuyển camera. Bộ pklot_v1 là giao thức của dự án, không phải bản sao split chính thức của PKLot.

| Giao thức | Train | Validation | Test |
| --- | --- | --- | --- |
| Trong camera | 1.182 | 270 | 270 |
| Chuyển UFPR04 → UFPR05 | 420 | 90 | 90 |

Các phương pháp dùng chung `TRAIN_FRAMES`, `VAL_FRAMES`, `TEST_FRAMES` và `SLOT_GEOMETRIES` từ cell dưới.


In [ ]:
split_candidates = [Path(globals().get("LOCKED_BUNDLE_ROOT", WORK_DIR / "outputs/locked_bundle")) / "data/manifests" / PROJECT_DATASET_VERSION,
                    WORK_DIR / "data/manifests" / PROJECT_DATASET_VERSION,
                    WORK_DIR / "splits" / PROJECT_DATASET_VERSION]
existing_split = next((path for path in split_candidates if (path / "split_lock.json").is_file()), None)
if globals().get("PREFER_LOCAL_FILES", True) and existing_split is not None:
    PROJECT_SPLIT_DIR = existing_split
    project_data = parking.load_project_splits(PROJECT_SPLIT_DIR)
    print("Dùng manifest đã khóa sẵn có; không tải lại metadata.")
else:
    PROJECT_SPLIT_DIR = WORK_DIR / "splits" / PROJECT_DATASET_VERSION
    project_data = parking.fetch_project_splits(OWNER, REPO, REF, PROJECT_SPLIT_DIR, dataset_version=PROJECT_DATASET_VERSION)
SLOT_GEOMETRIES = project_data["geometries"]
SPLIT_LOCK = project_data["lock"]
MANIFEST_DIR = PROJECT_SPLIT_DIR / PROTOCOL
TRAIN_FRAMES = project_data["protocols"][PROTOCOL]["train"]
VAL_FRAMES = project_data["protocols"][PROTOCOL]["val"]
TEST_FRAMES = project_data["protocols"][PROTOCOL]["test"]
print("Dataset / protocol:", PROJECT_DATASET_VERSION, PROTOCOL)
print("Train / validation / test:", len(TRAIN_FRAMES), len(VAL_FRAMES), len(TEST_FRAMES))
print("Geometry layouts:", len(SLOT_GEOMETRIES))
print("Manifest directory:", MANIFEST_DIR)
example_frame = parking.read_project_frame(TRAIN_FRAMES[0], DATASET_DIR)
print("Verified sample:", example_frame.sample_id)


## 4. Các phương pháp nhận diện


Ba phương pháp được so sánh trên cùng manifest đã khóa (mục 3.2) và chạy trong cùng một pipeline (mục 5):

| Mục | Thành phần | Vai trò |
|---|---|---|
| 4.1 | HOG + Linear SVM | Phân loại trực tiếp ảnh từng ô (crop theo polygon), học trên train. |
| 4.2 | Canny + Hough | Đoạn thẳng và mức hỗ trợ biên của từng ô. Mục 5 so với tham chiếu train để gắn cờ **hình học yếu**. Không dùng để kết luận ô trống. |
| 4.3 | ROI + tile màu | ROI bao các ô có padding, tối đa 2 × 2 tile chồng lấn, giữ offset/tỉ lệ/padding để quy đổi tọa độ. |
| 4.4 | YOLO11n full-frame | Phát hiện xe (car/bus/truck) trên toàn ảnh màu. |
| 4.5 | YOLO11n ROI/tile | Cùng detector trên từng tile, quy đổi box về ảnh gốc và chạy global NMS. |
| 4.6 | Ghép xe–ô | Dùng phần dưới box và polygon để xếp hạng, gán mỗi xe cho tối đa một ô, rồi suy ra trạng thái. Cùng một quy tắc cho full-frame và hybrid. |


### 4.1. HOG và SVM


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # ============================================================
    # 4.1. Baseline: HOG (scikit-image) + Linear SVM
    # Input : ảnh toàn cảnh + polygon các ô đỗ
    # Output: model và dự đoán trạng thái từng ô trên validation
    # Nhãn  : occupied=1, empty=0
    # ============================================================

    import importlib.util
    import subprocess
    import sys

    for module_name, package_name in [
        ("skimage", "scikit-image"),
        ("sklearn", "scikit-learn"),
        ("joblib", "joblib"),
        ("PIL", "Pillow"),
    ]:
        if importlib.util.find_spec(module_name) is None:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", package_name],
                check=True,
            )

    import csv
    import hashlib
    import json
    import time
    from pathlib import Path

    import joblib
    import numpy as np
    import PIL
    import skimage
    import sklearn
    from PIL import Image
    from skimage.feature import hog
    from sklearn.svm import LinearSVC

    required_variables = [
        "parking", "DATASET_DIR", "OUTPUT_DIR", "PROTOCOL",
        "PROJECT_DATASET_VERSION", "TRAIN_FRAMES", "VAL_FRAMES",
    ]
    missing = [name for name in required_variables if name not in globals()]
    if missing:
        raise RuntimeError(f"Chạy mục 3.2 trước. Thiếu biến: {missing}")

    if len(TRAIN_FRAMES) == 0 or len(VAL_FRAMES) == 0:
        raise ValueError("Train và validation phải có dữ liệu.")

    HOG_CONFIG = {
        "win_size": [64, 128],       # (width, height)
        "block_size": [16, 16],
        "block_stride": [8, 8],
        "cell_size": [8, 8],
        "nbins": 9,
    }

    SVM_C_VALUES = [0.01, 0.1, 1.0]
    HOG_OUTPUT_DIR = Path(OUTPUT_DIR) / PROTOCOL / "hog_svm"
    HOG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    class SkimageHOGExtractor:
        """Adapter HOG dùng scikit-image, không phụ thuộc OpenCV."""

        def __init__(self, config):
            self.config = {
                key: list(value) if isinstance(value, (list, tuple)) else value
                for key, value in config.items()
            }

            width, height = self.config["win_size"]
            cell_w, cell_h = self.config["cell_size"]
            block_w, block_h = self.config["block_size"]

            if min(width, height, cell_w, cell_h, block_w, block_h) <= 0:
                raise ValueError("Kích thước HOG phải dương.")
            if block_w % cell_w or block_h % cell_h:
                raise ValueError("block_size phải chia hết cho cell_size.")
            if tuple(self.config["block_stride"]) != tuple(
                self.config["cell_size"]
            ):
                raise ValueError(
                    "Adapter yêu cầu block_stride bằng cell_size."
                )

            cells_x, cells_y = width // cell_w, height // cell_h
            block_cells_x = block_w // cell_w
            block_cells_y = block_h // cell_h

            if cells_x < block_cells_x or cells_y < block_cells_y:
                raise ValueError("Ảnh resize nhỏ hơn một HOG block.")

            self.feature_size = (
                (cells_x - block_cells_x + 1)
                * (cells_y - block_cells_y + 1)
                * block_cells_x
                * block_cells_y
                * self.config["nbins"]
            )

        def getDescriptorSize(self):
            return self.feature_size

        def compute(self, image):
            config = self.config
            cell_w, cell_h = config["cell_size"]
            block_w, block_h = config["block_size"]

            features = hog(
                image,
                orientations=config["nbins"],
                pixels_per_cell=(cell_h, cell_w),
                cells_per_block=(
                    block_h // cell_h,
                    block_w // cell_w,
                ),
                block_norm="L2-Hys",
                transform_sqrt=False,
                feature_vector=True,
                channel_axis=None,
            )
            return np.asarray(features, dtype=np.float32)


    def make_hog(config):
        return SkimageHOGExtractor(config)


    HOG_EXTRACTOR = make_hog(HOG_CONFIG)


    def read_gray_frame(frame):
        """Đọc ảnh grayscale bằng Pillow."""
        image_path = Path(DATASET_DIR) / frame.image_path
        with Image.open(image_path) as image:
            gray = np.asarray(image.convert("L"))

        if gray.shape != (frame.height, frame.width):
            raise ValueError(f"Kích thước ảnh thay đổi: {frame.sample_id}")
        return gray


    def slot_hog_features(gray, polygon, extractor, config):
        """Cắt hình chữ nhật bao polygon ô đỗ rồi resize và trích HOG."""
        points = np.asarray(polygon, dtype=np.float32)
        if (
            points.ndim != 2
            or points.shape[1] != 2
            or len(points) < 3
            or not np.isfinite(points).all()
        ):
            raise ValueError(f"Polygon không hợp lệ: {polygon}")

        height, width = gray.shape
        x1, y1 = np.floor(points.min(axis=0)).astype(int)
        x2, y2 = np.ceil(points.max(axis=0)).astype(int)

        x1, x2 = max(0, x1), min(width, x2)
        y1, y2 = max(0, y1), min(height, y2)

        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"Polygon tạo crop rỗng: {polygon}")

        crop = Image.fromarray(gray[y1:y2, x1:x2])
        crop = crop.resize(
            tuple(config["win_size"]),
            resample=Image.Resampling.BILINEAR,
        )
        features = extractor.compute(np.asarray(crop))

        if features.size != extractor.getDescriptorSize():
            raise ValueError("Số chiều HOG không khớp cấu hình.")
        if not np.isfinite(features).all():
            raise ValueError("Feature HOG chứa NaN hoặc infinity.")

        return features


    # Kiểm tra extractor trước khi xử lý dataset.
    width, height = HOG_CONFIG["win_size"]
    dummy_features = HOG_EXTRACTOR.compute(
        np.zeros((height, width), dtype=np.uint8)
    )
    assert dummy_features.shape == (HOG_EXTRACTOR.getDescriptorSize(),)
    assert np.isfinite(dummy_features).all()
    print("HOG backend: scikit-image")
    print("Số chiều feature:", HOG_EXTRACTOR.getDescriptorSize())


    def build_hog_dataset(records, split_name):
        """
        Giữ nguyên split chung; không chia ngẫu nhiên ở cấp crop.
        Feature lưu bằng memmap. SVM vẫn cần RAM khi fit.
        """
        n_slots = sum(len(record["labels"]) for record in records)
        if n_slots == 0:
            raise ValueError(f"Tập {split_name} không có ô đỗ.")

        # Tên riêng cho mỗi lần chạy để tránh ghi đè memmap đang được dùng.
        feature_path = HOG_OUTPUT_DIR / (
            f"{split_name}_skimage_features_{time.time_ns()}.dat"
        )
        features = np.memmap(
            feature_path,
            dtype="float32",
            mode="w+",
            shape=(n_slots, HOG_EXTRACTOR.getDescriptorSize()),
        )
        labels = np.empty(n_slots, dtype=np.int32)
        metadata = []
        cursor = 0
        started = time.perf_counter()

        for index, record in enumerate(records):
            frame = parking.read_project_frame(record, DATASET_DIR)
            gray = read_gray_frame(frame)

            for space in frame.spaces:
                if cursor >= n_slots:
                    raise ValueError("Số ô thực tế vượt số ô trong manifest.")

                features[cursor] = slot_hog_features(
                    gray, space.polygon, HOG_EXTRACTOR, HOG_CONFIG
                )
                labels[cursor] = int(space.occupied)
                metadata.append({
                    "sample_id": frame.sample_id,
                    "space_id": space.space_id,
                    "camera": frame.camera,
                    "weather": frame.weather,
                    "date": frame.date,
                })
                cursor += 1

            if (index + 1) % 50 == 0 or index + 1 == len(records):
                print(
                    f"[{split_name}] {index + 1}/{len(records)} frames"
                    f" | {cursor} slots"
                )

        if cursor != n_slots:
            raise ValueError("Số ô thực tế không khớp manifest.")

        features.flush()
        print(
            f"[{split_name}] X={features.shape}, "
            f"empty={int((labels == 0).sum())}, "
            f"occupied={int((labels == 1).sum())}, "
            f"extraction={time.perf_counter() - started:.1f}s"
        )
        return features, labels, metadata


    # ---------- 1. Trích feature train và validation ----------
    X_train_hog, y_train_hog, _ = build_hog_dataset(
        TRAIN_FRAMES, "train"
    )
    X_val_hog, y_val_hog, HOG_VAL_METADATA = build_hog_dataset(
        VAL_FRAMES, "val"
    )

    if set(np.unique(y_train_hog)) != {0, 1}:
        raise ValueError("Train cần chứa cả ô trống và ô có xe.")

    # ---------- 2. Huấn luyện SVM, chọn C bằng F1 validation ----------
    # HOG đã chuẩn hóa theo block; baseline không thêm scaler.
    HOG_SEARCH_RESULTS = []
    best_f1 = -1.0
    HOG_SVM_MODEL = None

    for c_value in SVM_C_VALUES:
        candidate = LinearSVC(
            C=c_value,
            dual=False,
            max_iter=20000,
            random_state=42,
        )

        started = time.perf_counter()
        candidate.fit(X_train_hog, y_train_hog)
        fit_seconds = time.perf_counter() - started

        predictions = candidate.predict(X_val_hog).astype(int)
        metrics = parking.occupancy_metrics(
            y_val_hog.tolist(), predictions.tolist()
        )

        HOG_SEARCH_RESULTS.append({
            "C": c_value,
            "fit_seconds": fit_seconds,
            "n_iter": int(candidate.n_iter_),
            **metrics,
        })
        print(
            f"C={c_value:g} | "
            f"F1={metrics['f1']:.4f} | "
            f"Precision={metrics['precision']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
        )

        # Nếu hòa F1, giữ C nhỏ hơn do thử theo thứ tự tăng dần.
        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            HOG_SVM_MODEL = candidate

    # ---------- 3. Xuất dự đoán validation ----------
    HOG_VAL_MARGINS = HOG_SVM_MODEL.decision_function(X_val_hog)
    HOG_VAL_LABELS = HOG_SVM_MODEL.predict(X_val_hog).astype(int)

    # occupied_score là SVM margin, không phải xác suất.
    # Margin > 0 -> occupied; margin <= 0 -> empty.
    HOG_VAL_PREDICTIONS = [
        parking.OccupancyPrediction(
            sample_id=meta["sample_id"],
            space_id=meta["space_id"],
            occupied_score=float(margin),
        )
        for meta, margin in zip(HOG_VAL_METADATA, HOG_VAL_MARGINS)
    ]

    with (HOG_OUTPUT_DIR / "val_predictions.csv").open(
        "w", newline="", encoding="utf-8"
    ) as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=[
                "sample_id", "space_id", "camera", "weather", "date",
                "y_true", "y_pred", "occupied_score",
            ],
        )
        writer.writeheader()

        for meta, truth, prediction, margin in zip(
            HOG_VAL_METADATA,
            y_val_hog,
            HOG_VAL_LABELS,
            HOG_VAL_MARGINS,
        ):
            writer.writerow({
                **meta,
                "y_true": int(truth),
                "y_pred": int(prediction),
                "occupied_score": float(margin),
            })

    # ---------- 4. Lưu model và cấu hình ----------
    manifest_digest = hashlib.sha256(
        json.dumps(
            {"train": TRAIN_FRAMES, "val": VAL_FRAMES},
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()

    HOG_SVM_BUNDLE = {
        "model": HOG_SVM_MODEL,
        "hog_config": HOG_CONFIG,
        "hog_backend": "skimage",
        "resize_backend": "Pillow_BILINEAR",
        "crop_method": "axis_aligned_polygon_bounding_rectangle",
        "dataset_version": PROJECT_DATASET_VERSION,
        "protocol": PROTOCOL,
        "train_val_manifest_sha256": manifest_digest,
        "score_type": "svm_decision_margin",
        "decision_threshold": 0.0,
        "selected_C": float(HOG_SVM_MODEL.C),
        "versions": {
            "skimage": skimage.__version__,
            "sklearn": sklearn.__version__,
            "numpy": np.__version__,
            "Pillow": PIL.__version__,
        },
        "module_source": globals().get("provenance", {}),
    }

    joblib.dump(HOG_SVM_BUNDLE, HOG_OUTPUT_DIR / "hog_svm.joblib")

    (HOG_OUTPUT_DIR / "validation_search.json").write_text(
        json.dumps(HOG_SEARCH_RESULTS, indent=2),
        encoding="utf-8",
    )

    HOG_VAL_METRICS = parking.occupancy_metrics(
        y_val_hog.tolist(), HOG_VAL_LABELS.tolist()
    )
    (HOG_OUTPUT_DIR / "validation_metrics.json").write_text(
        json.dumps(HOG_VAL_METRICS, indent=2),
        encoding="utf-8",
    )


    # ---------- 5. Interface suy luận cho pipeline chung ----------
    def predict_hog_frame(frame, bundle=None):
        """
        Trả (list[OccupancyPrediction], elapsed_seconds).
        Thời gian gồm đọc ảnh, crop, resize, HOG và SVM;
        chưa gồm đọc XML hoặc khởi tạo extractor.
        Không sử dụng nhãn occupied khi suy luận.
        """
        bundle = HOG_SVM_BUNDLE if bundle is None else bundle

        if bundle.get("hog_backend") != "skimage":
            raise ValueError(
                "Model dùng backend HOG khác. Cần huấn luyện lại bằng skimage."
            )

        extractor = make_hog(bundle["hog_config"])
        started = time.perf_counter()

        if not frame.spaces:
            return [], time.perf_counter() - started

        gray = read_gray_frame(frame)
        features = np.stack([
            slot_hog_features(
                gray, space.polygon, extractor, bundle["hog_config"]
            )
            for space in frame.spaces
        ])
        margins = bundle["model"].decision_function(features)

        predictions = [
            parking.OccupancyPrediction(
                sample_id=frame.sample_id,
                space_id=space.space_id,
                occupied_score=float(margin),
            )
            for space, margin in zip(frame.spaces, margins)
        ]
        return predictions, time.perf_counter() - started


    print("\nSelected C:", HOG_SVM_MODEL.C)
    print("Validation:", json.dumps(HOG_VAL_METRICS, indent=2))
    print("Model / predictions:", HOG_OUTPUT_DIR)

    globals().get("sync_outputs_to_drive", lambda: None)()

### 4.2. Phân tích hình học bằng Canny và Hough

Hàm xử lý dùng chung nằm trong `parking_module.py`: `parking.analyze_parking_geometry(...)`. Notebook chỉ cấu hình, gọi hàm và trực quan hóa.

ảnh xám → Gaussian → Canny → HoughLinesP → lọc theo vùng bãi và hướng → đối chiếu polygon ô đỗ.

- Dùng polygon của đúng `geometry_id`, giữ tọa độ pixel ảnh gốc. Không resize, không dùng nhãn `labels`/`occupied` để phát hiện vạch.
- Vùng bãi là hợp các polygon, nới thêm một biên nhỏ. Chạy Canny trước khi áp mask để không tạo cạnh giả tại biên mask.
- Hướng được lấy từ các cạnh polygon, so sánh modulo 180°. Một đoạn chỉ được giữ nếu đủ tỷ lệ điểm nằm trong vùng bãi và gần một hướng hợp lệ.
- Kiểm tra từng cạnh bằng các điểm cách nhau tối đa 2 pixel: một điểm được hỗ trợ khi gần một đoạn **hữu hạn** và hướng đoạn gần hướng cạnh. Hợp bằng chứng từ nhiều đoạn, không cộng trùng các vạch do hai mép sơn.

`boundary_support` là tỷ lệ chiều dài chu vi có bằng chứng gần đó; `supported` nghĩa là đủ bằng chứng theo quy tắc bên dưới, **không phải** xác suất hình học đúng. `limited_support`/`no_support` không kết luận ô trống hay polygon sai. Xe, bóng, mưa và sơn mờ đều có thể che vạch; cạnh xe cũng có thể trùng vạch. Polygon XML không phải nhãn vạch sơn nên đây là kiểm tra tính nhất quán, không phải precision/recall phát hiện vạch.

Tài liệu thuật toán: [Canny](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html), [HoughLinesP](https://docs.opencv.org/4.x/d9/db0/tutorial_hough_lines.html).


In [ ]:
if "parking" not in globals() or not hasattr(parking, "analyze_parking_geometry"):
    raise RuntimeError(
        "Module chưa có hàm phân tích hình học. Push parking_module.py mới lên đúng OWNER/REPO/REF, "
        "rồi chạy lại cell tải/import module ở phần 2."
    )

# Sao chép mặc định từ module; chỉnh ở đây cho thí nghiệm hiện tại.
# Pixel trên ảnh gốc 1280x720; chỉ chọn tham số bằng train/validation của PROTOCOL.
# Đây là giá trị khởi đầu, chưa được xác nhận bằng thực nghiệm trên PKLot.
GEOMETRY_CONFIG = dict(parking.DEFAULT_GEOMETRY_CONFIG)
GEOMETRY_CONFIG.update({
    "canny_low": 50,
    "canny_high": 150,
    "hough_votes": 20,
    "min_line_length": 20,
})
GEOMETRY_DEMO_LIMIT = 9  # Tối đa một ảnh train cho mỗi camera/thời tiết.
print(json.dumps(GEOMETRY_CONFIG, indent=2))


In [ ]:
def show_parking_geometry(image_bgr, layout, result, title=""):
    """Hiện các bước cùng legend; mọi màu ảnh hiển thị theo RGB."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    overlay = rgb.copy()
    # Magenta = annotation, cyan = retained segment, green = boundary evidence.
    for space in layout["spaces"]:
        polygon = np.rint(space["polygon"]).astype(np.int32)
        cv2.polylines(overlay, [polygon], True, (220, 70, 210), 1, cv2.LINE_AA)
    for line in result["segments"]:
        x1, y1, x2, y2 = line["xyxy"]
        color = (50, 220, 100) if line["supports_boundary"] else (0, 200, 255)
        cv2.line(overlay, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
    fig, axes = plt.subplots(2, 3, figsize=(16, 7), layout="constrained")
    panels = [(rgb, "Original"), (result["images"]["gray"], "Grayscale"),
              (result["images"]["blurred"], "Gaussian"),
              (result["images"]["edges"], "Canny (full image)"),
              (result["images"]["roi_edges"], "Canny inside parking ROI"),
              (overlay, f"Lines: {result['raw_line_count']} raw / {len(result['segments'])} retained")]
    for axis, (image, label) in zip(axes.flat, panels):
        axis.imshow(image, cmap="gray" if image.ndim == 2 else None,
                    vmin=0 if image.ndim == 2 else None, vmax=255 if image.ndim == 2 else None)
        axis.set_title(label)
        axis.axis("off")
    axes[1, 2].legend(handles=[
        Line2D([0], [0], color="#dc46d2", label="XML slot boundary"),
        Line2D([0], [0], color="#00c8ff", label="Retained line"),
        Line2D([0], [0], color="#32dc64", label="Supports slot boundary"),
    ], loc="lower right", fontsize=8, framealpha=0.9)
    fig.suptitle(title, fontsize=11)
    return fig


**Chạy minh họa và lưu kết quả:** chọn tối đa một ảnh **train** cho mỗi camera/thời tiết, luân phiên camera. Đây chỉ là kiểm tra trực quan, chưa phải đánh giá toàn bộ tập dữ liệu. `GEOMETRY_RESULTS` chứa kết quả gọn cho bước sau, theo `sample_id`; không chứa dự đoán occupancy. Các ảnh trung gian chỉ giữ cho frame đang xem để giảm bộ nhớ.

Mỗi lần chạy xuất `geometry_results.json` dưới `OUTPUT_DIR/<PROTOCOL>/geometry/`, gồm tham số, phiên bản thư viện, REF/SHA module, SHA manifest và ID hình học. Chỉ ghi kết quả cá nhân, không ghi dataset hoặc manifest chung. Muốn chạy lại sau khi chỉnh cấu hình, chạy lại các cell từ phần cấu hình trở xuống.


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Bỏ demo hình học/ROI; runner 6.2 hoặc 6.3 tự thực hiện khi cần.")
else:
    geometry_required = ("TRAIN_FRAMES", "SLOT_GEOMETRIES", "DATASET_DIR", "OUTPUT_DIR", "PROTOCOL")
    if any(name not in globals() for name in geometry_required):
        raise RuntimeError("Chạy phần 3.2 để tải manifest và hình học trước phần phân tích hình học.")
    if type(GEOMETRY_DEMO_LIMIT) is not int or GEOMETRY_DEMO_LIMIT < 1:
        raise ValueError("GEOMETRY_DEMO_LIMIT phải là số nguyên dương")

    # Chọn bằng metadata, không xem nhãn hoặc chọn theo kết quả phát hiện.
    geometry_groups = {}
    for record in sorted(TRAIN_FRAMES, key=lambda row: row["sample_id"]):
        if record["split"] != "train":
            raise ValueError("Geometry demo expects training frames")
        geometry_groups.setdefault((record["camera"], record["weather"]), record)
    geometry_examples = [geometry_groups[key] for key in sorted(geometry_groups, key=lambda k: (k[1], k[0]))]
    geometry_examples = geometry_examples[:GEOMETRY_DEMO_LIMIT]
    if not geometry_examples:
        raise ValueError("TRAIN_FRAMES không có frame để minh họa")

    GEOMETRY_RESULTS = {}
    for record in geometry_examples:
        layout = SLOT_GEOMETRIES[record["geometry_id"]]
        if layout["camera"] != record["camera"]:
            raise ValueError("Geometry belongs to another camera")
        # Kiểm tra nhãn/layout vẫn khớp bản khóa; hàm phát hiện không nhận nhãn.
        frame = parking.read_project_frame(record, DATASET_DIR)
        image_path = Path(DATASET_DIR) / frame.image_path
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Không đọc được ảnh: {image_path}")
        result = parking.analyze_parking_geometry(image, layout, config=GEOMETRY_CONFIG)
        compact = {key: value for key, value in result.items() if key != "images"}
        compact.update(sample_id=record["sample_id"], geometry_id=record["geometry_id"],
                       camera=record["camera"], weather=record["weather"], split=record["split"])
        GEOMETRY_RESULTS[record["sample_id"]] = compact
        figure = show_parking_geometry(image, layout, result,
                                       f"{record['camera']} / {record['weather']} / {record['sample_id']}")
        plt.show()
        plt.close(figure)
        print(
            f"{record['camera']} / {record['weather']} - "
            f"{len(result['segments'])} retained segments; "
            f"slot flags: {result['quality_counts']}."
        )

    GEOMETRY_OUTPUT_DIR = Path(OUTPUT_DIR) / PROTOCOL / "geometry"
    GEOMETRY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    geometry_export = {
        "task": "parking_line_geometry", "protocol": PROTOCOL, "split": "train",
        "interpretation": "Boundary evidence only; not occupancy or calibrated accuracy",
        "config": dict(GEOMETRY_CONFIG),
        "versions": {"opencv": cv2.__version__, "numpy": np.__version__,
                     "matplotlib": mpl.__version__},
        "module_source": globals().get("provenance", {}),
        "manifest_sha256": globals().get("SPLIT_LOCK", {}).get("artifacts_sha256", {}),
        "results": list(GEOMETRY_RESULTS.values()),
    }
    geometry_output_file = GEOMETRY_OUTPUT_DIR / "geometry_results.json"
    geometry_output_file.write_text(json.dumps(geometry_export, indent=2, ensure_ascii=False) + "\n")
    print("Geometry output:", geometry_output_file)

    globals().get("sync_outputs_to_drive", lambda: None)()

**Cách đọc và bàn giao:** `segments` chứa `xyxy`, chiều dài, góc và cờ có hỗ trợ biên hay không. Mỗi ô trong `slots` có `edge_support` theo thứ tự cạnh contour, `boundary_support` có trọng số chiều dài, số cạnh đủ hỗ trợ và `quality_flag`. Mặc định, một cạnh cần ≥25% bằng chứng và một ô cần ≥2 cạnh để có cờ `supported`; thiếu bằng chứng giữ cờ `limited_support` hoặc `no_support`. Các ngưỡng này là giá trị khởi đầu, chưa xác nhận trên PKLot.

Kiểm tra ảnh trung gian và overlay trên các camera/thời tiết; sau đó chọn tham số bằng train/validation. Khi chạy giao thức chuyển góc nhìn, chỉ chọn tham số trên UFPR04 của giao thức đó, không chỉnh theo UFPR05 test. Không dùng accuracy/F1 occupancy cho đầu ra hình học này. Bước suy luận trạng thái ô ở mục 4.6 cần bằng chứng khác; không chuyển `no_support` thành nhãn trống.


### 4.3. ROI và chia tile

Lấy ROI bao các polygon của đúng `geometry_id`, thêm padding rồi chia tối đa 2×2 tile **màu BGR**. Mỗi ô và vùng ngữ cảnh quanh ô phải nằm trọn trong ít nhất một tile. Overlap được tăng theo hình học; nếu vượt giới hạn thì dùng một ROI đầy đủ. Trục quá nhỏ cũng không chia tiếp. Canny/Hough không tạo vùng ô mới.

`parking.make_parking_tiles(...)` trả ảnh màu, metadata và bảng kiểm tra bao phủ. Mặc định giữ pixel gốc; `output_size=(640, 640)` bật resize giữ tỉ lệ và letterbox. `offset_xy`, `scale_xy`, `padding_ltrb` mô tả phép biến đổi thực tế sau làm tròn kích thước. `tile_points_to_image`, `image_points_to_tile`, `tile_boxes_to_image` quy đổi tọa độ. Bounds dùng `[x1,y1,x2,y2]`, với cạnh phải/dưới là điểm kết thúc của lát cắt.


In [ ]:
if "parking" not in globals() or not hasattr(parking, "make_parking_tiles"):
    raise RuntimeError("Cập nhật module từ GitHub bằng cell tải/import ở phần 2 trước khi chạy ROI.")

TILE_CONFIG = {
    "padding": 32,           # Ngữ cảnh mỗi ô, pixel gốc, cắt tại biên ảnh.
    "grid": (2, 2),          # (số hàng, số cột), mỗi chiều tối đa 2.
    "overlap": 0.20,         # Bề rộng phần giao / kích thước ROI trên mỗi trục chia.
    "max_overlap": 0.60,     # Nếu cần nhiều hơn, trả một ROI thay vì cắt mất ô.
    "min_tile_size": 256,    # Trục ngắn hơn 2 lần giá trị này dùng một tile.
    "output_size": None,    # Hoặc (640, 640) để chuẩn bị đầu vào detector.
}
TILE_DEMO_LIMIT = 3


In [ ]:
def show_parking_tiles(image_bgr, layout, result, title=""):
    panel_count = 1 + len(result["tiles"])
    panel_rows = (panel_count + 1) // 2
    figure, axes = plt.subplots(panel_rows, 2, figsize=(14, 4 * panel_rows),
                                layout="constrained", squeeze=False)
    axes = axes.ravel()
    axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    for space in layout["spaces"]:
        polygon = np.asarray(space["polygon"])
        closed = np.vstack([polygon, polygon[0]])
        axes[0].plot(closed[:, 0], closed[:, 1], color="magenta", linewidth=0.7)
    left, top, right, bottom = result["roi_bounds_xyxy"]
    axes[0].add_patch(plt.Rectangle((left, top), right-left, bottom-top,
                                    fill=False, edgecolor="yellow", linewidth=2, label="Parking ROI"))
    for tile in result["tiles"]:
        metadata = tile["metadata"]
        left, top, right, bottom = metadata["bounds_xyxy"]
        axes[0].add_patch(plt.Rectangle((left, top), right-left, bottom-top,
                                        fill=False, edgecolor="cyan", linestyle="--", linewidth=1))
        axes[0].text(left+4, top+14, metadata["tile_id"], color="black", fontsize=9,
                     bbox={"facecolor": "cyan", "alpha": 0.85, "pad": 1})
    axes[0].set_title(f"ROI and grid {result['grid']} | overlap {result['effective_overlap']:.2f}")
    axes[0].legend(loc="lower right")
    for axis, tile in zip(axes[1:], result["tiles"]):
        metadata = tile["metadata"]
        axis.imshow(cv2.cvtColor(tile["image"], cv2.COLOR_BGR2RGB))
        axis.set_title(f"{metadata['tile_id']} | offset {metadata['offset_xy']} | size {metadata['output_size']}")
        # Full slot polygons in tile coordinates show context and boundary coverage.
        for space in layout["spaces"]:
            if space["space_id"] in metadata["space_ids"]:
                polygon = parking.image_points_to_tile(space["polygon"], metadata)
                closed = np.vstack([polygon, polygon[0]])
                axis.plot(closed[:, 0], closed[:, 1], color="magenta", linewidth=0.7)
    for axis in axes:
        axis.axis("off")
    figure.suptitle(title, fontsize=11)
    return figure


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Bỏ demo hình học/ROI; runner 6.2 hoặc 6.3 tự thực hiện khi cần.")
else:
    if any(name not in globals() for name in ("TRAIN_FRAMES", "SLOT_GEOMETRIES", "DATASET_DIR")):
        raise RuntimeError("Chạy phần 3.2 để tải manifest và layout trước khi tạo ROI.")
    if type(TILE_DEMO_LIMIT) is not int or TILE_DEMO_LIMIT < 1:
        raise ValueError("TILE_DEMO_LIMIT phải là số nguyên dương")

    # Một ảnh train cho mỗi camera, chọn theo metadata, không dùng nhãn occupancy.
    tile_examples = {}
    for record in sorted(TRAIN_FRAMES, key=lambda row: row["sample_id"]):
        if record["split"] != "train":
            raise ValueError("ROI demo expects training frames")
        tile_examples.setdefault(record["camera"], record)
    if not tile_examples:
        raise ValueError("TRAIN_FRAMES không có ảnh minh họa")

    TILE_RESULTS = {}
    for record in list(tile_examples.values())[:TILE_DEMO_LIMIT]:
        frame = parking.read_project_frame(record, DATASET_DIR)
        layout = SLOT_GEOMETRIES[record["geometry_id"]]
        if layout["camera"] != record["camera"]:
            raise ValueError("Geometry belongs to another camera")
        image = cv2.imread(str(Path(DATASET_DIR) / frame.image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(frame.image_path)
        tile_result = parking.make_parking_tiles(image, layout, **TILE_CONFIG)
        tile_result.update(sample_id=record["sample_id"], geometry_id=record["geometry_id"])
        TILE_RESULTS[record["sample_id"]] = tile_result
        figure = show_parking_tiles(image, layout, tile_result,
                                    f"{record['camera']} / {record['weather']} / {record['sample_id']}")
        plt.show()
        plt.close(figure)
        print(f"{record['camera']}: {len(tile_result['tiles'])} color tiles; "
              f"all spaces covered={tile_result['all_spaces_covered']}; "
              f"context covered={tile_result['all_context_covered']}; "
              f"fallback={tile_result['fallback_reason']}")
        # Ví dụ phép đổi tọa độ: 4 góc crop phải trở về đúng bounds gốc.
        for tile in tile_result["tiles"]:
            metadata = tile["metadata"]
            left, top, right, bottom = metadata["bounds_xyxy"]
            corners = np.array([[left, top], [right, top], [right, bottom], [left, bottom]])
            tile_corners = parking.image_points_to_tile(corners, metadata)
            np.testing.assert_allclose(parking.tile_points_to_image(tile_corners, metadata), corners, atol=1e-8)

    # TILE_RESULTS giữ tile["image"] màu BGR và tile["metadata"] cho bước detector.
    # Khi có boxes xyxy từ detector trên tile:
    # boxes_on_original = parking.tile_boxes_to_image(boxes_on_tile, tile["metadata"], clip=True)
    # Cần loại box diện tích 0 sau clipping và ghép detection trùng ở vùng overlap ở bước detector.


### 4.4. YOLO trên toàn ảnh


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # ============================================================
    # 4.4. YOLO11n pretrained COCO trên ảnh toàn cảnh
    # Input : ảnh màu toàn cảnh
    # Output: list[parking.VehicleDetection], tọa độ ảnh gốc
    # Chọn imgsz trên validation; chưa chạy test hoặc fine-tune.
    # ============================================================

    import importlib.util
    import subprocess
    import sys

    if importlib.util.find_spec("ultralytics") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "ultralytics"],
            check=True,
        )

    import hashlib
    import json
    import time
    from dataclasses import asdict
    from pathlib import Path

    import cv2
    import numpy as np
    import torch
    import ultralytics
    from ultralytics import YOLO

    assert VAL_FRAMES, "Chạy mục 3.2 trước."

    # ---------- Checkpoint YOLO11n: tải có thử lại, lưu cục bộ và trên Drive ----------
    # GitHub Releases thỉnh thoảng trả 504 trong vài chục giây; trình tải của Ultralytics chỉ thử 3 lần.
    YOLO_CHECKPOINT_NAME = "yolo11n.pt"
    YOLO_CHECKPOINT_URLS = [
        f"https://github.com/ultralytics/assets/releases/download/{tag}/{YOLO_CHECKPOINT_NAME}"
        for tag in ("v8.4.0", "v8.3.0", "v8.2.0")
    ]
    YOLO_CHECKPOINT_MIN_BYTES = 1_000_000
    YOLO_CHECKPOINT_PATH = OUTPUT_DIR.parent.parent / "models" / YOLO_CHECKPOINT_NAME   # đĩa cục bộ khi dùng Drive
    YOLO_CHECKPOINT_DRIVE = WORK_DIR / "models" / YOLO_CHECKPOINT_NAME


    def ensure_yolo_checkpoint(rounds=4):
        path = YOLO_CHECKPOINT_PATH
        if path.is_file() and path.stat().st_size >= YOLO_CHECKPOINT_MIN_BYTES:
            return path
        path.parent.mkdir(parents=True, exist_ok=True)
        if YOLO_CHECKPOINT_DRIVE != path and YOLO_CHECKPOINT_DRIVE.is_file()             and YOLO_CHECKPOINT_DRIVE.stat().st_size >= YOLO_CHECKPOINT_MIN_BYTES:
            shutil.copyfile(YOLO_CHECKPOINT_DRIVE, path)
            print("Dùng checkpoint đã lưu trên Drive:", YOLO_CHECKPOINT_DRIVE)
            return path
        last_error = None
        for round_index in range(rounds):
            for url in YOLO_CHECKPOINT_URLS:
                temp = path.with_suffix(".part")
                try:
                    with urllib.request.urlopen(url, timeout=60) as response, temp.open("wb") as stream:
                        shutil.copyfileobj(response, stream)
                    if temp.stat().st_size < YOLO_CHECKPOINT_MIN_BYTES:
                        raise OSError(f"File tải về quá nhỏ ({temp.stat().st_size} B)")
                    temp.replace(path)
                    print("Đã tải checkpoint:", url)
                    if YOLO_CHECKPOINT_DRIVE != path and USE_GOOGLE_DRIVE:
                        try:
                            _ensure_drive_dir(YOLO_CHECKPOINT_DRIVE.parent)
                            shutil.copyfile(path, YOLO_CHECKPOINT_DRIVE)
                        except (OSError, RuntimeError) as error:
                            print("  ⚠ Chưa lưu được checkpoint lên Drive (không ảnh hưởng chạy tiếp):", error)
                    return path
                except (OSError, urllib.error.URLError) as error:
                    last_error = error
                    temp.unlink(missing_ok=True)
                    print(f"  ⚠ Tải thất bại ({error.__class__.__name__}: {error}) từ {url}")
            wait = 15 * (round_index + 1)
            print(f"  Thử lại vòng {round_index + 2}/{rounds} sau {wait}s")
            time.sleep(wait)
        raise RuntimeError(
            "Không tải được yolo11n.pt từ GitHub. Tải thủ công và đặt vào "
            f"{YOLO_CHECKPOINT_DRIVE} (hoặc {YOLO_CHECKPOINT_PATH}) rồi chạy lại cell. Lỗi cuối: {last_error}"
        )


    YOLO_CHECKPOINT_PATH = ensure_yolo_checkpoint()
    print("YOLO checkpoint:", YOLO_CHECKPOINT_PATH, f"({YOLO_CHECKPOINT_PATH.stat().st_size:,} B)")

    YOLO_CONFIG = {
        "checkpoint": str(YOLO_CHECKPOINT_PATH),
        "imgsz": 640,                # Giá trị khởi đầu; chọn lại ở bước 3.
        "conf": 0.25,
        "iou": 0.70,                 # Ngưỡng NMS bên trong YOLO.
        "max_det": 300,
        "vehicle_classes": ["car", "bus", "truck"],
        "device": 0 if torch.cuda.is_available() else "cpu",
    }

    YOLO_OUTPUT_DIR = OUTPUT_DIR / PROTOCOL / "yolo_fullframe"
    YOLO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    class FullFrameYOLODetector:
        """
        Tương thích parking.VehicleDetector:
            predict(image_path) -> list[VehicleDetection]

        OpenCV đọc ảnh BGR; Ultralytics hỗ trợ trực tiếp ndarray BGR.
        result.boxes.xyxy đã nằm trong tọa độ ảnh đầu vào gốc.
        """

        def __init__(self, config):
            self.config = dict(config)
            self.model = YOLO(self.config["checkpoint"])

            names = self.model.names
            self.class_ids = [
                int(class_id)
                for class_id, name in names.items()
                if name in self.config["vehicle_classes"]
            ]
            found = {names[class_id] for class_id in self.class_ids}
            missing = set(self.config["vehicle_classes"]) - found
            if missing:
                raise ValueError(f"Checkpoint thiếu các lớp: {missing}")

            self.last_timing = {}

        def _sync(self):
            if str(self.config["device"]) != "cpu":
                torch.cuda.synchronize(self.config["device"])

        def _infer(self, image):
            return self.model.predict(
                source=image,
                imgsz=self.config["imgsz"],
                conf=self.config["conf"],
                iou=self.config["iou"],
                classes=self.class_ids,
                max_det=self.config["max_det"],
                device=self.config["device"],
                augment=False,
                agnostic_nms=False,
                verbose=False,
                save=False,
            )[0]

        def warmup(self):
            """Loại chi phí khởi tạo khỏi phép đo trên validation."""
            size = self.config["imgsz"]
            self._infer(np.zeros((size, size, 3), dtype=np.uint8))
            self._sync()

        def predict(self, image_path):
            self._sync()
            started = time.perf_counter()

            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                raise FileNotFoundError(f"Không đọc được ảnh: {image_path}")

            return self.predict_image(image, started=started)

        def predict_image(self, image, started=None):
            """Như predict nhưng nhận ảnh BGR đã đọc (pipeline mục 5 đọc ảnh một lần)."""
            self._sync()
            read_done = time.perf_counter()
            if started is None:
                started = read_done
            result = self._infer(image)
            self._sync()
            inference_done = time.perf_counter()

            detections = []
            if result.boxes is not None:
                coordinates = result.boxes.xyxy.cpu().numpy()
                confidences = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)

                height, width = image.shape[:2]

                for box, confidence, class_id in zip(
                    coordinates, confidences, classes
                ):
                    x1, y1, x2, y2 = map(float, box)
                    x1, x2 = np.clip([x1, x2], 0, width)
                    y1, y2 = np.clip([y1, y2], 0, height)

                    if x2 <= x1 or y2 <= y1:
                        continue

                    detections.append(
                        parking.VehicleDetection(
                            xyxy=(float(x1), float(y1), float(x2), float(y2)),
                            confidence=float(confidence),
                            class_name=self.model.names[int(class_id)],
                        )
                    )

            finished = time.perf_counter()
            self.last_timing = {
                "read_seconds": read_done - started,
                # Bao gồm preprocessing, forward và NMS của Ultralytics.
                "predict_seconds": inference_done - read_done,
                "convert_seconds": finished - inference_done,
                "total_seconds": finished - started,
                "inference_calls": 1,
                # Số đo nội bộ do Ultralytics báo, đơn vị milliseconds.
                "ultralytics_ms": {
                    key: float(value) for key, value in result.speed.items()
                },
            }
            return detections


    # ---------- 1. Khởi tạo và warm-up ----------
    FULLFRAME_DETECTOR = FullFrameYOLODetector(YOLO_CONFIG)
    FULLFRAME_DETECTOR.warmup()

    print("Device:", YOLO_CONFIG["device"])
    print("Vehicle classes:", YOLO_CONFIG["vehicle_classes"])
    print("Ultralytics:", ultralytics.__version__)

    # ---------- 2. Checkpoint để tái lập ----------
    checkpoint_path = Path(
        getattr(FULLFRAME_DETECTOR.model, "ckpt_path", None)
        or YOLO_CONFIG["checkpoint"]
    )
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_path}")

    checkpoint_sha256 = hashlib.sha256(
        checkpoint_path.read_bytes()
    ).hexdigest()

    # ---------- 3. Chọn imgsz trên validation ----------
    # Ở 640, xe nhỏ ở xa (PUCPR) thường bị bỏ sót. Mỗi ứng viên chạy toàn bộ
    # VAL_FRAMES; chấm bằng quy tắc ghép xe–ô mặc định (parking.infer_parking_occupancy)
    # và evaluator chung. Chọn F1 occupancy cao nhất; hòa thì imgsz nhỏ hơn (nhanh hơn).
    # Test không được dùng. Hybrid (4.5) dùng cùng detector nên dùng cùng imgsz.
    # Rút ngắn danh sách (ví dụ [640]) nếu cần tiết kiệm thời gian Colab.
    # Rút ngắn thời gian: YOLO_IMGSZ_CANDIDATES = [640]
    YOLO_IMGSZ_CANDIDATES = [640, 960, 1280]

    def run_fullframe_validation(imgsz):
        FULLFRAME_DETECTOR.config["imgsz"] = int(imgsz)
        FULLFRAME_DETECTOR.warmup()
        detections_by_sample, timings = {}, []
        for index, record in enumerate(VAL_FRAMES):
            frame = parking.read_project_frame(record, DATASET_DIR)
            detections_by_sample[frame.sample_id] = FULLFRAME_DETECTOR.predict(
                Path(DATASET_DIR) / frame.image_path
            )
            timings.append(dict(FULLFRAME_DETECTOR.last_timing))
            if (index + 1) % 50 == 0 or index + 1 == len(VAL_FRAMES):
                print(f"[YOLO full-frame imgsz={imgsz}] {index + 1}/{len(VAL_FRAMES)}")
        predictions = [
            p
            for record in VAL_FRAMES
            for p in parking.infer_parking_occupancy(
                record["sample_id"], detections_by_sample[record["sample_id"]],
                SLOT_GEOMETRIES[record["geometry_id"]],
            )["predictions"]
        ]
        overall = parking.evaluate_occupancy(
            VAL_FRAMES, predictions, missing_policy="error"
        )["overall"]
        return detections_by_sample, timings, overall


    imgsz_search, imgsz_runs = [], {}
    for imgsz in sorted(set(YOLO_IMGSZ_CANDIDATES)):
        detections_by_sample, timings, overall = run_fullframe_validation(imgsz)
        imgsz_runs[imgsz] = (detections_by_sample, timings)
        imgsz_search.append({
            "imgsz": imgsz,
            "f1": overall["f1"],
            "coverage": overall["coverage"],
            "mae_vacant_count": overall["mae_vacant_count"],
            "detections_per_frame": float(np.mean([len(d) for d in detections_by_sample.values()])),
            "mean_seconds_per_frame": float(np.mean([t["total_seconds"] for t in timings])),
        })
        print(f"imgsz={imgsz}: occupancy F1={overall['f1']:.4f}, "
              f"{imgsz_search[-1]['detections_per_frame']:.1f} box/ảnh, "
              f"{imgsz_search[-1]['mean_seconds_per_frame']:.3f}s/ảnh")

    best = max(imgsz_search, key=lambda row: (round(row["f1"], 6), -row["imgsz"]))
    YOLO_CONFIG["imgsz"] = FULLFRAME_DETECTOR.config["imgsz"] = best["imgsz"]
    YOLO_FULL_VAL, YOLO_FULL_TIMINGS = imgsz_runs[best["imgsz"]]  # Không chạy lại.
    del imgsz_runs
    print("Chọn imgsz =", best["imgsz"], "(dùng chung cho full-frame và hybrid)")

    run_info = {
        "config": YOLO_CONFIG,
        "checkpoint_sha256": checkpoint_sha256,
        "dataset_version": PROJECT_DATASET_VERSION,
        "protocol": PROTOCOL,
        "split": "val",
        "val_manifest_sha256": hashlib.sha256(
            json.dumps(VAL_FRAMES, sort_keys=True).encode("utf-8")
        ).hexdigest(),
        "imgsz_selection": {
            "candidates": sorted(set(YOLO_IMGSZ_CANDIDATES)),
            "rule": "max occupancy F1 on val with parking.DEFAULT_OCCUPANCY_CONFIG; tie -> smaller imgsz",
            "results": imgsz_search,
        },
        "versions": {
            "ultralytics": ultralytics.__version__,
            "torch": torch.__version__,
            "opencv": cv2.__version__,
            "numpy": np.__version__,
        },
        "timing_scope": (
            "Image reading + preprocessing + model + NMS + conversion. "
            "Excludes XML audit, warm-up, visualization and JSON writing."
        ),
    }

    (YOLO_OUTPUT_DIR / "run_config.json").write_text(
        json.dumps(run_info, indent=2), encoding="utf-8"
    )

    # Lưu detections validation của imgsz đã chọn.
    prediction_path = YOLO_OUTPUT_DIR / "val_detections.jsonl"
    with prediction_path.open("w", encoding="utf-8") as stream:
        for record, timing in zip(VAL_FRAMES, YOLO_FULL_TIMINGS):
            stream.write(json.dumps({
                "sample_id": record["sample_id"],
                "image_path": record["image_path"],
                "camera": record["camera"],
                "weather": record["weather"],
                "date": record["date"],
                "width": record["width"],
                "height": record["height"],
                "detections": [asdict(item) for item in YOLO_FULL_VAL[record["sample_id"]]],
                "timing": timing,
            }) + "\n")

    # ---------- 4. Lưu thống kê thời gian ----------
    times = np.array([
        item["total_seconds"] for item in YOLO_FULL_TIMINGS
    ])

    timing_summary = {
        "num_frames": len(times),
        "mean_seconds_per_frame": float(times.mean()),
        "median_seconds_per_frame": float(np.median(times)),
        "p95_seconds_per_frame": float(np.percentile(times, 95)),
        "total_measured_seconds": float(times.sum()),
    }

    (YOLO_OUTPUT_DIR / "timing_summary.json").write_text(
        json.dumps(timing_summary, indent=2), encoding="utf-8"
    )

    print("\nTiming:", json.dumps(timing_summary, indent=2))
    print("Predictions:", prediction_path)

    # ---------- 5. Hiển thị box trên một ảnh validation ----------
    import matplotlib.pyplot as plt

    demo_frame = parking.read_project_frame(VAL_FRAMES[0], DATASET_DIR)
    demo_image = cv2.imread(
        str(Path(DATASET_DIR) / demo_frame.image_path), cv2.IMREAD_COLOR
    )

    for detection in YOLO_FULL_VAL[demo_frame.sample_id]:
        x1, y1, x2, y2 = map(int, detection.xyxy)
        cv2.rectangle(demo_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            demo_image,
            f"{detection.class_name} {detection.confidence:.2f}",
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )

    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(demo_image, cv2.COLOR_BGR2RGB))
    plt.title("YOLO11n — full-frame vehicle detections")
    plt.axis("off")
    plt.show()

    globals().get("sync_outputs_to_drive", lambda: None)()

### 4.5. Kết hợp ROI với YOLO


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # ============================================================
    # 4.5. Hybrid: ROI/tile + cùng YOLO11n + global NMS
    #
    # Yêu cầu: đã chạy mục 3.2, 4.3 và 4.4.
    # Output : HYBRID_VAL[sample_id] -> list[VehicleDetection]
    #
    # Dùng tiler và cấu hình chung ở phần 4.3, gồm kiểm tra bao phủ/ngữ cảnh.
    # Ultralytics trả box trong tọa độ ảnh tile đầu vào.
    # Mapper chung xử lý offset, resize và padding của tile về ảnh gốc.
    # ============================================================

    assert "FULLFRAME_DETECTOR" in globals(), "Chạy cell 4.4 trước."
    assert "TILE_CONFIG" in globals(), "Chạy cell cấu hình 4.3 trước."

    HYBRID_CONFIG = {
        "tiling": dict(TILE_CONFIG),  # Cùng cấu hình và cách tính overlap với phần 4.3.
        "global_nms_iou": 0.50,
        "edge_margin_px": 4,
        "min_box_size_px": 2,
    }

    HYBRID_OUTPUT_DIR = OUTPUT_DIR / PROTOCOL / "yolo_hybrid"
    HYBRID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Dùng chính model/checkpoint/cấu hình của phần 4.4.
    HYBRID_DETECTOR = FULLFRAME_DETECTOR


    def box_iou(box, others):
        """IoU giữa một box và mảng các box xyxy."""
        left_top = np.maximum(box[:2], others[:, :2])
        right_bottom = np.minimum(box[2:], others[:, 2:])
        intersection = np.maximum(right_bottom - left_top, 0).prod(axis=1)

        box_area = np.maximum(box[2:] - box[:2], 0).prod()
        other_areas = np.maximum(
            others[:, 2:] - others[:, :2], 0
        ).prod(axis=1)

        return intersection / np.maximum(
            box_area + other_areas - intersection, 1e-12
        )


    def global_vehicle_nms(candidates, iou_threshold):
        """
        NMS chung giữa các lớp phương tiện:
        một xe có thể bị dự đoán car ở tile này và truck ở tile khác.

        Ưu tiên box không chạm biên crop, sau đó confidence.
        Box chạm biên vẫn được giữ nếu không bị một box khác loại.
        """
        if not candidates:
            return []

        boxes = np.asarray([item["xyxy"] for item in candidates], dtype=float)
        order = sorted(
            range(len(candidates)),
            key=lambda i: (
                candidates[i]["touches_crop_edge"],
                -candidates[i]["confidence"],
            ),
        )

        keep = []
        while order:
            current = order[0]
            keep.append(current)
            remaining = np.asarray(order[1:], dtype=int)
            if len(remaining) == 0:
                break

            overlaps = box_iou(boxes[current], boxes[remaining])
            order = remaining[overlaps <= iou_threshold].tolist()

        return [candidates[i] for i in keep]


    def predict_hybrid_frame(frame, image=None):
        """
        Trả:
          detections : list[parking.VehicleDetection]
          timing     : thời gian đọc ảnh, tạo tile, YOLO và gộp box
          metadata   : tile và thông tin box giữ lại

        Không dùng nhãn occupied để suy luận.
        """
        detector = HYBRID_DETECTOR
        detector._sync()
        started = time.perf_counter()

        if image is None:
            # Pipeline mục 5 truyền ảnh đã đọc để không đọc lại.
            image_path = DATASET_DIR / frame.image_path
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                raise FileNotFoundError(f"Không đọc được ảnh: {image_path}")

        height, width = image.shape[:2]
        if (height, width) != (frame.height, frame.width):
            raise ValueError(f"Kích thước ảnh thay đổi: {frame.sample_id}")

        read_done = time.perf_counter()
        layout = {
            "width": frame.width, "height": frame.height,
            "spaces": [{"space_id": space.space_id, "polygon": space.polygon}
                       for space in frame.spaces],
        }
        tile_result = parking.make_parking_tiles(image, layout, **HYBRID_CONFIG["tiling"])
        tiles = tile_result["tiles"]
        tiles_done = time.perf_counter()

        candidates = []
        predict_seconds = 0.0
        mapping_seconds = 0.0
        internal_ms = {"preprocess": 0.0, "inference": 0.0, "postprocess": 0.0}

        for tile in tiles:
            detector._sync()
            infer_started = time.perf_counter()

            # _infer dùng đúng imgsz, conf, classes, iou... của phần 4.4.
            result = detector._infer(tile["image"])
            detector._sync()
            infer_done = time.perf_counter()
            predict_seconds += infer_done - infer_started

            for key in internal_ms:
                internal_ms[key] += float(result.speed.get(key, 0.0))

            tile_metadata = tile["metadata"]
            left, top, right, bottom = tile_metadata["bounds_xyxy"]

            if result.boxes is not None:
                boxes = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)

                for box, score, class_id in zip(boxes, scores, classes):
                    # Đổi từ ảnh tile đầu vào về ảnh gốc đúng một lần.
                    # Clipping loại phần box nằm trong letterbox padding.
                    x1, y1, x2, y2 = parking.tile_boxes_to_image(
                        box, tile_metadata, clip=True
                    )

                    if min(x2 - x1, y2 - y1) < HYBRID_CONFIG["min_box_size_px"]:
                        continue

                    margin = HYBRID_CONFIG["edge_margin_px"]

                    # Chỉ gắn cờ biên do crop; biên ảnh gốc không tính.
                    touches_edge = bool(
                        (left > 0 and x1 <= left + margin)
                        or (top > 0 and y1 <= top + margin)
                        or (right < width and x2 >= right - margin)
                        or (bottom < height and y2 >= bottom - margin)
                    )

                    candidates.append({
                        "xyxy": (
                            float(x1), float(y1), float(x2), float(y2),
                        ),
                        "confidence": float(score),
                        "class_name": detector.model.names[int(class_id)],
                        "tile_id": tile_metadata["tile_id"],
                        "touches_crop_edge": touches_edge,
                    })

            mapping_seconds += time.perf_counter() - infer_done

        nms_started = time.perf_counter()
        kept = global_vehicle_nms(
            candidates, HYBRID_CONFIG["global_nms_iou"]
        )

        detections = [
            parking.VehicleDetection(
                xyxy=item["xyxy"],
                confidence=item["confidence"],
                class_name=item["class_name"],
            )
            for item in kept
        ]

        finished = time.perf_counter()
        timing = {
            "read_seconds": read_done - started,
            "tile_seconds": tiles_done - read_done,
            "predict_seconds": predict_seconds,
            "mapping_seconds": mapping_seconds,
            "global_nms_and_conversion_seconds": finished - nms_started,
            "total_seconds": finished - started,
            "inference_calls": len(tiles),
            "ultralytics_ms_sum": internal_ms,
        }

        metadata = {
            "tiling": {key: value for key, value in tile_result.items() if key != "tiles"},
            "tiles": [
                # xyxy giữ tương thích với các cell vẽ tile hiện có.
                {**tile["metadata"], "xyxy": tile["metadata"]["bounds_xyxy"]}
                for tile in tiles
            ],
            "boxes_before_nms": len(candidates),
            "boxes_after_nms": len(kept),
            "kept_box_sources": [
                {
                    "tile_id": item["tile_id"],
                    "touches_crop_edge": item["touches_crop_edge"],
                }
                for item in kept
            ],
        }
        return detections, timing, metadata


    # ---------- 1. Ghi cấu hình ----------
    hybrid_run_info = {
        "hybrid_config": HYBRID_CONFIG,
        "tiling_implementation": "parking.make_parking_tiles",
        "detector_config": dict(HYBRID_DETECTOR.config),
        "checkpoint_sha256": checkpoint_sha256,   # Từ cell 4.4.
        "versions": run_info["versions"],
        "dataset_version": PROJECT_DATASET_VERSION,
        "protocol": PROTOCOL,
        "split": "val",
        "val_manifest_sha256": run_info["val_manifest_sha256"],
        "edge_policy": "prefer non-crop-edge boxes, then confidence",
        "global_nms_class_agnostic": True,
        "timing_scope": (
            "Image reading + tile creation + all YOLO calls + mapping + "
            "global NMS. Excludes XML audit, warm-up, JSON and visualization."
        ),
    }

    (HYBRID_OUTPUT_DIR / "run_config.json").write_text(
        json.dumps(hybrid_run_info, indent=2), encoding="utf-8"
    )

    # ---------- 2. Warm-up và chạy cùng validation ----------
    HYBRID_DETECTOR.warmup()

    HYBRID_VAL = {}
    HYBRID_TIMINGS = []
    HYBRID_METADATA = {}

    prediction_path = HYBRID_OUTPUT_DIR / "val_detections.jsonl"

    with prediction_path.open("w", encoding="utf-8") as stream:
        for index, record in enumerate(VAL_FRAMES):
            frame = parking.read_project_frame(record, DATASET_DIR)
            detections, timing, metadata = predict_hybrid_frame(frame)

            HYBRID_VAL[frame.sample_id] = detections
            HYBRID_TIMINGS.append(timing)
            HYBRID_METADATA[frame.sample_id] = metadata

            stream.write(json.dumps({
                "sample_id": frame.sample_id,
                "image_path": frame.image_path,
                "camera": frame.camera,
                "weather": frame.weather,
                "date": frame.date,
                "width": frame.width,
                "height": frame.height,
                "detections": [asdict(item) for item in detections],
                "timing": timing,
                "metadata": metadata,
            }) + "\n")

            if (index + 1) % 50 == 0 or index + 1 == len(VAL_FRAMES):
                print(f"[Hybrid] {index + 1}/{len(VAL_FRAMES)}")

    # ---------- 3. Thống kê thời gian ----------
    hybrid_times = np.asarray([
        timing["total_seconds"] for timing in HYBRID_TIMINGS
    ])

    hybrid_summary = {
        "num_frames": len(hybrid_times),
        "mean_seconds_per_frame": float(hybrid_times.mean()),
        "median_seconds_per_frame": float(np.median(hybrid_times)),
        "p95_seconds_per_frame": float(np.percentile(hybrid_times, 95)),
        "total_measured_seconds": float(hybrid_times.sum()),
        "total_inference_calls": sum(
            timing["inference_calls"] for timing in HYBRID_TIMINGS
        ),
    }

    (HYBRID_OUTPUT_DIR / "timing_summary.json").write_text(
        json.dumps(hybrid_summary, indent=2), encoding="utf-8"
    )

    print("\nHybrid timing:", json.dumps(hybrid_summary, indent=2))
    print("Predictions:", prediction_path)

    # ---------- 4. So sánh trực quan trên cùng một frame ----------
    demo_frame = parking.read_project_frame(VAL_FRAMES[0], DATASET_DIR)
    original = cv2.imread(
        str(DATASET_DIR / demo_frame.image_path), cv2.IMREAD_COLOR
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    for ax, title, detections, show_tiles in [
        (
            axes[0], "YOLO full-frame",
            YOLO_FULL_VAL[demo_frame.sample_id], False,
        ),
        (
            axes[1], "YOLO ROI/tiles + global NMS",
            HYBRID_VAL[demo_frame.sample_id], True,
        ),
    ]:
        canvas = original.copy()

        if show_tiles:
            for tile in HYBRID_METADATA[demo_frame.sample_id]["tiles"]:
                x1, y1, x2, y2 = tile["xyxy"]
                cv2.rectangle(
                    canvas, (x1, y1), (x2 - 1, y2 - 1), (255, 180, 0), 2
                )
                cv2.putText(
                    canvas, f"Tile {tile['tile_id']}",
                    (x1 + 5, y1 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 180, 0), 2,
                )

        for detection in detections:
            x1, y1, x2, y2 = map(int, detection.xyxy)
            cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                canvas,
                f"{detection.class_name} {detection.confidence:.2f}",
                (x1, max(15, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1,
            )

        ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{title} - {len(detections)} detections")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    globals().get("sync_outputs_to_drive", lambda: None)()

In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # ============================================================
    # Thử YOLO full-frame và hybrid trên một ảnh validation khác
    # ============================================================

    from pathlib import Path
    import cv2
    import matplotlib.pyplot as plt

    # None: tự chọn ảnh có ít nhất một ô occupied theo nhãn.
    # Hoặc nhập chỉ số từ 0 đến len(VAL_FRAMES)-1, ví dụ 50.
    IMAGE_INDEX = None

    if IMAGE_INDEX is None:
        IMAGE_INDEX = next(
            (
                i for i, record in enumerate(VAL_FRAMES)
                if any(int(label) == 1 for label in record["labels"].values())
            ),
            None,
        )
        if IMAGE_INDEX is None:
            raise ValueError("Không tìm thấy ảnh validation có ô occupied.")

    if not 0 <= IMAGE_INDEX < len(VAL_FRAMES):
        raise ValueError(
            f"IMAGE_INDEX phải nằm trong [0, {len(VAL_FRAMES) - 1}]."
        )

    frame = parking.read_project_frame(
        VAL_FRAMES[IMAGE_INDEX], DATASET_DIR
    )
    image_path = Path(DATASET_DIR) / frame.image_path

    # Chạy lại detector trên ảnh đã chọn.
    full_detections = FULLFRAME_DETECTOR.predict(image_path)
    full_timing = dict(FULLFRAME_DETECTOR.last_timing)

    hybrid_detections, hybrid_timing, tile_metadata = (
        predict_hybrid_frame(frame)
    )

    print("Validation index:", IMAGE_INDEX)
    print("Sample:", frame.sample_id)
    print("Camera / weather:", frame.camera, "/", frame.weather)
    print("Số ô được gán nhãn có xe:", sum(
        int(space.occupied) for space in frame.spaces
    ))
    print(
        f"Full-frame: {len(full_detections)} detections"
        f" | {full_timing['total_seconds']:.3f}s"
    )
    print(
        f"Hybrid: {len(hybrid_detections)} detections"
        f" | {hybrid_timing['total_seconds']:.3f}s"
        f" | {hybrid_timing['inference_calls']} lượt suy luận"
    )
    print(
        "Hybrid trước/sau global NMS:",
        tile_metadata["boxes_before_nms"],
        "/",
        tile_metadata["boxes_after_nms"],
    )

    original = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if original is None:
        raise FileNotFoundError(image_path)

    fig, axes = plt.subplots(1, 2, figsize=(22, 10))

    panels = [
        (axes[0], "YOLO full-frame", full_detections, False),
        (axes[1], "YOLO ROI/tiles + NMS", hybrid_detections, True),
    ]

    for ax, title, detections, show_tiles in panels:
        canvas = original.copy()

        if show_tiles:
            for tile in tile_metadata["tiles"]:
                x1, y1, x2, y2 = tile["xyxy"]
                cv2.rectangle(
                    canvas, (x1, y1), (x2 - 1, y2 - 1),
                    (255, 180, 0), 2,
                )
                cv2.putText(
                    canvas,
                    f"Tile {tile['tile_id']}",
                    (x1 + 5, y1 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (255, 180, 0), 2,
                )

        for detection in detections:
            x1, y1, x2, y2 = map(int, detection.xyxy)
            cv2.rectangle(
                canvas, (x1, y1), (x2, y2), (0, 255, 0), 2
            )
            cv2.putText(
                canvas,
                f"{detection.class_name} {detection.confidence:.2f}",
                (x1, max(15, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (0, 255, 0), 1, cv2.LINE_AA,
            )

        ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{title} — {len(detections)} detections")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    from pathlib import Path
    import cv2
    import matplotlib.pyplot as plt

    image_path = Path(DATASET_DIR) / frame.image_path
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(image_path)

    print("Sample:", frame.sample_id)
    print("Kích thước ảnh gốc:", image.shape)
    print("Cấu hình detector:", FULLFRAME_DETECTOR.config)
    print("Class IDs:", FULLFRAME_DETECTOR.class_ids)

    experiments = [
        ("Full-frame: 640, conf=0.25", 640, 0.25),
        ("Full-frame: 1280, conf=0.25", 1280, 0.25),
        ("Full-frame: 640, conf=0.05", 640, 0.05),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    config = FULLFRAME_DETECTOR.config

    for ax, (title, image_size, confidence) in zip(axes, experiments):
        result = FULLFRAME_DETECTOR.model.predict(
            source=image.copy(),
            imgsz=image_size,
            conf=confidence,
            iou=config["iou"],
            classes=FULLFRAME_DETECTOR.class_ids,
            max_det=config["max_det"],
            device=config["device"],
            augment=False,
            agnostic_nms=False,
            verbose=False,
            save=False,
        )[0]

        count = len(result.boxes) if result.boxes is not None else 0
        print(f"{title}: {count} detections")

        annotated = result.plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{title}\n{count} detections")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### 4.6. Ghép xe với ô đỗ và suy luận trạng thái


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # Cùng quy tắc cho full-frame và hybrid, trong tọa độ ảnh gốc.
    # Cấu hình mặc định chỉ để xem trước; chọn và khóa trên validation ở 6.2.
    # Không có box phù hợp được coi là trống: detector bỏ sót xe có thể gây sai.
    # score là điểm ghép hình học, không phải xác suất occupied đã hiệu chuẩn.
    OCCUPANCY_CONFIG = dict(parking.DEFAULT_OCCUPANCY_CONFIG)
    OCCUPANCY_OUTPUT_DIR = Path(OUTPUT_DIR) / PROTOCOL / "occupancy"
    OCCUPANCY_DETECTIONS = {"yolo_fullframe": YOLO_FULL_VAL, "yolo_hybrid": HYBRID_VAL}


    def run_occupancy(config):
        expected = {record["sample_id"] for record in VAL_FRAMES}
        if not expected or len(expected) != len(VAL_FRAMES):
            raise ValueError("Validation manifest is empty or contains duplicate samples")
        results = {}
        for method, detections in OCCUPANCY_DETECTIONS.items():
            if set(detections) != expected:
                raise ValueError(f"{method}: detections must cover exactly the current validation split")
            results[method] = {}
            for record in VAL_FRAMES:
                sample_id = record["sample_id"]
                result = parking.infer_parking_occupancy(
                    sample_id, detections[sample_id], SLOT_GEOMETRIES[record["geometry_id"]], config
                )
                results[method][sample_id] = result
        return results


    def export_occupancy(results, directory):
        directory.mkdir(parents=True, exist_ok=True)
        for method, frames in results.items():
            method_dir = directory / method
            method_dir.mkdir(parents=True, exist_ok=True)
            predictions = [p for result in frames.values() for p in result["predictions"]]
            (method_dir / "predictions.jsonl").write_text(
                "".join(json.dumps(p) + "\n" for p in predictions), encoding="utf-8"
            )
            (method_dir / "frames.json").write_text(json.dumps(frames, indent=2), encoding="utf-8")


    def plot_occupancy(results, record):
        image = cv2.imread(str(DATASET_DIR / record["image_path"]), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(record["image_path"])
        sample_id = record["sample_id"]
        spaces = SLOT_GEOMETRIES[record["geometry_id"]]["spaces"]
        colors = {"occupied": "red", "vacant": "limegreen", "unknown": "orange"}
        fig, axes = plt.subplots(1, 2, figsize=(20, 9))
        for ax, (method, frames) in zip(axes, results.items()):
            result = frames[sample_id]
            predictions = {p["slot_id"]: p for p in result["predictions"]}
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            for index in result["kept_detection_indices"]:
                x1, y1, x2, y2 = OCCUPANCY_DETECTIONS[method][sample_id][index].xyxy
                ax.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], color="deepskyblue", lw=1)
            for space in spaces:
                prediction = predictions[str(space["space_id"])]
                polygon = np.asarray(space["polygon"])
                closed = np.vstack([polygon, polygon[0]])
                center = polygon.mean(axis=0)
                color = colors[prediction["state"]]
                ax.plot(closed[:, 0], closed[:, 1], color=color, lw=1.5)
                ax.text(*center, str(space["space_id"]), color=color, fontsize=7)
                index = prediction["detection_index"]
                if index is not None:
                    x1, y1, x2, y2 = OCCUPANCY_DETECTIONS[method][sample_id][index].xyxy
                    lower_center = ((x1+x2)/2, y2-(y2-y1)*result["config"]["lower_fraction"]/2)
                    ax.plot([center[0], lower_center[0]], [center[1], lower_center[1]], color="white", lw=1)
            counts = result["counts"]
            ax.set_title(f"{method}: occupied={counts['occupied']}, vacant={counts['vacant']}, unknown={counts['unknown']}")
            ax.legend(handles=[Line2D([0], [0], color=color, label=state) for state, color in colors.items()])
            ax.axis("off")
        plt.tight_layout()
        plt.show()


    OCCUPANCY_PREVIEW = run_occupancy(OCCUPANCY_CONFIG)
    export_occupancy(OCCUPANCY_PREVIEW, OCCUPANCY_OUTPUT_DIR / "preview")
    print("Cấu hình ghép ở đây chỉ để xem trước; cấu hình chính thức được chọn trên validation ở mục 6.2.")
    print("Preview outputs:", OCCUPANCY_OUTPUT_DIR / "preview")
    plot_occupancy(OCCUPANCY_PREVIEW, VAL_FRAMES[0])

    globals().get("sync_outputs_to_drive", lambda: None)()

## 5. Pipeline và trực quan hóa


`run_pipeline_frame(record, method)` xử lý một ảnh gốc theo đúng thứ tự dưới đây và đo thời gian bằng `parking.FrameTimer`. Ba phương pháp `hog_svm`, `yolo_fullframe`, `yolo_hybrid` dùng chung các bước; chỉ bước model khác nhau.

| Bước | Giai đoạn đo | Hàm (mục) |
|---|---|---|
| Đọc ảnh BGR, kiểm tra kích thước | preprocess | OpenCV |
| Canny/Hough, mức hỗ trợ biên | preprocess | `parking.analyze_parking_geometry` (4.2) |
| Cờ hình học yếu so với tham chiếu train | preprocess | `parking.assess_geometry_quality` |
| ROI/tile → YOLO từng tile → quy đổi box → global NMS | model | `predict_hybrid_frame` (4.3, 4.5) |
| YOLO toàn ảnh | model | `FULLFRAME_DETECTOR.predict_image` (4.4) |
| HOG + SVM từng ô | model | `predict_hog_frame` (4.1) |
| Ghép xe–ô, trạng thái | postprocess | `parking.infer_parking_occupancy` (4.6) |
| Hình học yếu → không báo trống | postprocess | `parking.apply_geometry_policy` |

**Cờ hình học yếu.** Thử trên ảnh PKLot thật, chỉ khoảng 40–65% ô có cờ `supported` của 4.2 ngay cả trên ảnh bình thường. Lý do là vạch sơn không luôn trùng cạnh polygon XML. Vì vậy, các cờ tuyệt đối không đủ để đánh giá một ảnh có đáng tin hay không. Mục 5 dựng **tham chiếu theo `geometry_id` từ tối đa 15 ảnh train** (ảnh median, tỉ lệ `supported` của từng ô, số đoạn thẳng), không dùng nhãn. Các cờ được gắn như sau:

- `layout_shift_suspected`: phase correlation với ảnh tham chiếu lệch quá 6 px, nghĩa là camera có thể đã dịch chuyển.
- `few_line_segments`: số đoạn thẳng ít hơn 25% median tham chiếu (tối, mờ, mưa, bị che).
- `boundary_lost` (từng ô): ô thường `supported` trong tham chiếu nhưng ảnh này mất vạch.

Khi có cờ, pipeline in `⚠ [Hình học yếu] ...`. Ô bị 4.6 báo **vacant** sẽ thành **unknown**: không tự coi là trống. Ô **occupied** giữ nguyên vì phát hiện được xe là bằng chứng dương. Layout không có ảnh train (UFPR05 trong giao thức chuyển camera) không thể so sánh và được ghi `reference_available=False`.

Thử trên 48 ảnh thật (đầu archive PKLot), cờ frame bật trên 0–4,5% ảnh gốc, 100% ảnh dịch 10 px, 64–91% ảnh dịch 7 px và 100% ảnh giảm độ sáng còn 30%. Cờ ô bật trên 2–10% ô của ảnh gốc.

Cell dưới dựng tham chiếu, chạy cả ba phương pháp trên một ảnh validation, vẽ 6 panel (ảnh gốc, Canny, Hough + ô cờ yếu, ROI/tile, box xe, ô tô màu) và xuất CSV/JSON demo. Cấu hình ghép ở demo là bản xem trước của 4.6; mục 6.2 chốt lại.


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # ============================================================
    # 5. Pipeline tích hợp
    #   đọc ảnh -> hình học (4.2) + đánh giá chất lượng so với tham chiếu
    #   -> ROI/tile (4.3) -> detector (4.4/4.5) hoặc HOG/SVM (4.1)
    #   -> quy đổi box về ảnh gốc (4.5) -> ghép xe–ô + trạng thái (4.6)
    #   -> cờ hình học yếu: không báo "trống" -> đếm, cảnh báo, CSV/JSON.
    #
    # Pipeline chỉ GỌI lại các hàm đã có ở mục 4 và trong parking_module.py:
    #   parking.analyze_parking_geometry, parking.assess_geometry_quality,
    #   FULLFRAME_DETECTOR.predict_image, predict_hybrid_frame (dùng
    #   parking.make_parking_tiles + parking.tile_boxes_to_image + global NMS),
    #   parking.infer_parking_occupancy, predict_hog_frame.
    # Thời gian đo bằng parking.FrameTimer trên ẢNH GỐC (gồm mọi tile).
    # ============================================================

    import csv
    import json
    import time
    from dataclasses import asdict
    from pathlib import Path

    required = ["parking", "DATASET_DIR", "OUTPUT_DIR", "PROTOCOL", "TRAIN_FRAMES", "VAL_FRAMES",
                "SLOT_GEOMETRIES", "GEOMETRY_CONFIG", "TILE_CONFIG", "FULLFRAME_DETECTOR",
                "predict_hybrid_frame", "OCCUPANCY_CONFIG", "HOG_SVM_BUNDLE", "predict_hog_frame"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(f"Chạy các mục 3.2 và 4.1–4.6 trước. Thiếu: {missing}")
    for name in ("build_geometry_reference", "assess_geometry_quality", "apply_geometry_policy"):
        if not hasattr(parking, name):
            raise RuntimeError("Module cũ: push parking_module.py mới rồi chạy lại cell tải module ở mục 2.")

    PIPELINE_METHODS = ("hog_svm", "yolo_fullframe", "yolo_hybrid")
    METHOD_LABELS = {"hog_svm": "HOG + SVM", "yolo_fullframe": "YOLO full-frame",
                     "yolo_hybrid": "YOLO ROI/tile"}
    STATE_COLORS = {"vacant": "limegreen", "occupied": "red", "unknown": "orange"}  # Như mục 4.6.
    PIPELINE_OUTPUT_DIR = Path(OUTPUT_DIR) / PROTOCOL / "pipeline"
    PIPELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Cờ hình học yếu so với tham chiếu train của cùng layout (không dùng nhãn).
    GEOMETRY_QUALITY_CONFIG = dict(parking.DEFAULT_GEOMETRY_QUALITY_CONFIG)
    GEOMETRY_REFERENCE_FRAMES = 15         # Tối đa số ảnh train cho mỗi geometry_id.
    WEAK_GEOMETRY_POLICY = "unknown"       # Bắt buộc: không tự coi ô là trống khi hình học yếu.
    GEOMETRY_FLAG_MESSAGES = {
        "layout_shift_suspected": "ảnh lệch so với tham chiếu của layout, camera có thể đã dịch chuyển",
        "few_line_segments": "quá ít đoạn thẳng so với tham chiếu (tối, mờ, mưa hoặc bị che)",
        "boundary_lost": "vạch quanh ô thường thấy rõ nhưng không tìm thấy ở ảnh này",
    }


    def retry_io(action, retries=5, delay=1.5):
        """
        Chạy action() (ghi file) và thử lại khi gặp OSError. Google Drive trên Colab đôi khi báo
        FileNotFoundError ngay sau khi tạo thư mục hoặc ghi liên tiếp nhiều file. Action phải
        ghi đè an toàn (idempotent). Lỗi thật vẫn được ném lại sau lần thử cuối.
        """
        for attempt in range(retries):
            try:
                return action()
            except OSError as error:
                if attempt == retries - 1:
                    raise
                print(f"  ⚠ Ghi file lỗi ({error.__class__.__name__}: {error}); thử lại {attempt + 1}/{retries - 1}")
                time.sleep(delay * (attempt + 1))


    def write_bytes_retry(path, data):
        path = Path(path)

        def action():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)

        retry_io(action)
        return path


    def save_figure(fig, path, dpi=80):
        """Render PNG vào bộ nhớ rồi ghi file có thử lại (không phụ thuộc đường dẫn Drive khi render)."""
        import io
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
        return write_bytes_retry(path, buffer.getvalue())


    def read_bgr(frame):
        path = Path(DATASET_DIR) / frame.image_path
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Không đọc được ảnh: {path}")
        if image.shape[:2] != (frame.height, frame.width):
            raise ValueError(f"Kích thước ảnh thay đổi: {frame.sample_id}")
        return image


    # ---------- Tham chiếu hình học từ ảnh TRAIN ----------
    def fit_geometry_references(records, max_frames=GEOMETRY_REFERENCE_FRAMES):
        """Chọn đều tối đa max_frames ảnh train cho mỗi layout; chỉ dùng ảnh, không dùng nhãn."""
        by_layout = {}
        for record in records:
            by_layout.setdefault(record["geometry_id"], []).append(record)
        references = {}
        for gid, rows in sorted(by_layout.items()):
            rows = sorted(rows, key=lambda r: r["sample_id"])
            count = min(len(rows), max_frames)
            chosen = [rows[int(i)] for i in np.linspace(0, len(rows) - 1, count).round()]
            images = (read_bgr(parking.read_project_frame(r, DATASET_DIR)) for r in chosen)
            references[gid] = parking.build_geometry_reference(
                images, SLOT_GEOMETRIES[gid], GEOMETRY_CONFIG, GEOMETRY_QUALITY_CONFIG)
        return references


    started = time.perf_counter()
    GEOMETRY_REFERENCES = fit_geometry_references(TRAIN_FRAMES)
    print(f"Tham chiếu hình học: {len(GEOMETRY_REFERENCES)} layout ({time.perf_counter() - started:.1f}s)")
    reference_dir = PIPELINE_OUTPUT_DIR / "geometry_reference"
    reference_dir.mkdir(parents=True, exist_ok=True)
    for gid, reference in GEOMETRY_REFERENCES.items():
        checkable = sum(s["supported_rate"] >= GEOMETRY_QUALITY_CONFIG["reliable_supported_rate"]
                        for s in reference["slots"].values())
        print(f"  {gid}: {reference['frames']} ảnh train | ô kiểm được bằng vạch: "
              f"{checkable}/{len(reference['slots'])} | median đoạn thẳng: {reference['segment_count_median']:.0f}")
        write_bytes_retry(reference_dir / f"{gid}.png", cv2.imencode(".png", reference["gray"])[1].tobytes())
    (reference_dir / "references.json").write_text(json.dumps(
        {gid: {k: v for k, v in ref.items() if k != "gray"} for gid, ref in GEOMETRY_REFERENCES.items()},
        indent=2), encoding="utf-8")
    missing_reference = sorted({r["geometry_id"] for r in VAL_FRAMES + list(TEST_FRAMES or [])}
                               - set(GEOMETRY_REFERENCES))
    if missing_reference:
        print("  ⚠ Không có ảnh train cho layout", missing_reference,
              "-> bỏ qua kiểm tra lệch camera/mất vạch (reference_available=False).")


    def geometry_messages(sample_id, quality):
        messages = [f"[Hình học yếu] {sample_id}: {GEOMETRY_FLAG_MESSAGES[flag]}"
                    + (f" (dx={quality['camera_shift']['dx']:+.1f}, dy={quality['camera_shift']['dy']:+.1f} px)"
                       if flag == "layout_shift_suspected" else "")
                    for flag in quality["frame_flags"]]
        if quality["weak_slot_ids"]:
            ids = quality["weak_slot_ids"]
            messages.append(f"[Hình học yếu] {sample_id}: {len(ids)} ô {GEOMETRY_FLAG_MESSAGES['boundary_lost']} "
                            f"(ví dụ {', '.join(ids[:6])})")
        return messages


    def hog_predictions(frame):
        """HOG/SVM của 4.1: margin > 0 -> occupied (ngưỡng đã chốt ở 4.1)."""
        raw, _ = predict_hog_frame(frame)
        threshold = HOG_SVM_BUNDLE.get("decision_threshold", 0.0)
        return [{"sample_id": p.sample_id, "slot_id": p.space_id, "score": p.occupied_score,
                 "state": "occupied" if p.occupied_score > threshold else "vacant",
                 "detection_index": None, "match_score": None} for p in raw]


    def run_pipeline_frame(record, method, occupancy_config=None, keep_images=False):
        """
        Một ảnh gốc qua toàn bộ pipeline. Trả dict gồm:
          raw_predictions (trước chính sách hình học), predictions (sau chính sách),
          detections, tiles, geometry (gọn), quality, warnings, counts, timing.
        Không đọc nhãn occupied để suy luận.
        """
        if method not in PIPELINE_METHODS:
            raise ValueError(f"method phải thuộc {PIPELINE_METHODS}")
        occupancy_config = OCCUPANCY_CONFIG if occupancy_config is None else occupancy_config
        frame = parking.read_project_frame(record, DATASET_DIR)
        layout = SLOT_GEOMETRIES[record["geometry_id"]]
        sync = (lambda: None) if method == "hog_svm" else FULLFRAME_DETECTOR._sync
        timer = parking.FrameTimer(frame.sample_id, synchronize=sync)
        detections, tiles, kept, calls = [], [], [], 0
        with timer:
            with timer.stage("preprocess"):
                image = read_bgr(frame)
                geometry = parking.analyze_parking_geometry(image, layout, GEOMETRY_CONFIG)
                quality = parking.assess_geometry_quality(
                    geometry, GEOMETRY_REFERENCES.get(record["geometry_id"]), GEOMETRY_QUALITY_CONFIG)
            with timer.stage("model"):
                if method == "hog_svm":
                    raw_predictions = hog_predictions(frame)
                elif method == "yolo_fullframe":
                    detections = FULLFRAME_DETECTOR.predict_image(image)
                    calls = 1
                else:
                    # Tile, YOLO trên từng tile, quy đổi box về ảnh gốc và global NMS (4.5).
                    detections, _, hybrid_meta = predict_hybrid_frame(frame, image=image)
                    tiles = hybrid_meta["tiles"]
                    calls = len(tiles)
            with timer.stage("postprocess"):
                if method != "hog_svm":
                    occupancy = parking.infer_parking_occupancy(
                        frame.sample_id, detections, layout, occupancy_config)
                    raw_predictions, kept = occupancy["predictions"], occupancy["kept_detection_indices"]
                policy = parking.apply_geometry_policy(raw_predictions, quality, WEAK_GEOMETRY_POLICY)
                predictions = policy["predictions"]
                counts = {s: sum(p["state"] == s for p in predictions) for s in STATE_COLORS}

        warnings_ = geometry_messages(frame.sample_id, quality)
        if policy["changed_space_ids"]:
            warnings_.append(f"[Hình học yếu] {frame.sample_id}: {len(policy['changed_space_ids'])} ô "
                             f"không có xe được phát hiện nhưng hình học yếu -> UNKNOWN, không tính là trống.")
        result = {
            "record": record, "frame": frame, "method": method, "layout": layout,
            "occupancy_config": dict(occupancy_config) if method != "hog_svm" else None,
            "raw_predictions": raw_predictions, "predictions": predictions,
            "geometry_unknown_space_ids": policy["changed_space_ids"],
            "detections": detections, "kept_detection_indices": kept, "tiles": tiles,
            "geometry": {k: v for k, v in geometry.items() if k != "images"},
            "quality": quality, "warnings": warnings_, "counts": counts,
            "timing": timer.record(inference_calls=calls),
        }
        if keep_images:
            result["image_bgr"] = image
            result["geometry_images"] = geometry["images"]
        return result


    def rescore_result(result, occupancy_config=None, policy=WEAK_GEOMETRY_POLICY):
        """Tính lại trạng thái với cấu hình ghép/chính sách khác, KHÔNG chạy lại model."""
        raw = result["raw_predictions"]
        if result["method"] != "hog_svm" and occupancy_config is not None:
            raw = parking.infer_parking_occupancy(result["frame"].sample_id, result["detections"],
                                                  result["layout"], occupancy_config)["predictions"]
        return parking.apply_geometry_policy(raw, result["quality"], policy)["predictions"]


    def run_pipeline(records, method, occupancy_config=None, label="", max_frames=None,
                     print_every=50, max_printed_warnings=8):
        """Chạy cả split; in tiến độ và tóm tắt cảnh báo hình học."""
        if method != "hog_svm":
            FULLFRAME_DETECTOR.warmup()  # Ngoài phép đo.
        records = list(records)[:max_frames] if max_frames else list(records)
        results, printed, weak_frames, changed = [], 0, 0, 0
        for index, record in enumerate(records):
            result = run_pipeline_frame(record, method, occupancy_config)
            results.append(result)
            weak_frames += bool(result["quality"]["frame_flags"])
            changed += len(result["geometry_unknown_space_ids"])
            for message in result["warnings"]:
                if printed < max_printed_warnings:
                    print("  ⚠", message)
                printed += 1
            if (index + 1) % print_every == 0 or index + 1 == len(records):
                print(f"[{label or method}] {index + 1}/{len(records)}")
        if printed > max_printed_warnings:
            print(f"  ... tổng cộng {printed} thông báo hình học (xem warnings.txt)")
        print(f"[{label or method}] frame có cờ hình học yếu: {weak_frames}/{len(records)}; "
              f"ô vacant -> unknown do hình học yếu: {changed}")
        return results


    # ---------- Xuất CSV/JSON ----------
    SLOT_CSV_FIELDS = ["sample_id", "space_id", "camera", "weather", "date", "method", "state",
                       "raw_state", "score", "match_score", "detection_index", "geometry_flags",
                       "boundary_support", "geometry_quality_flag", "polygon"]


    def frame_record(result, predictions=None):
        """JSON gọn của một ảnh: đếm, cảnh báo, hình học, tile, box, timing, từng ô."""
        frame, predictions = result["frame"], predictions or result["predictions"]
        polygons = {str(s["space_id"]): s["polygon"] for s in result["layout"]["spaces"]}
        geometry_slots = {s["space_id"]: s for s in result["geometry"]["slots"]}
        raw = {p["slot_id"]: p["state"] for p in result["raw_predictions"]}
        counts = {s: sum(p["state"] == s for p in predictions) for s in STATE_COLORS}
        return {
            "sample_id": frame.sample_id, "image_path": frame.image_path, "camera": frame.camera,
            "weather": frame.weather, "date": frame.date, "width": frame.width, "height": frame.height,
            "geometry_id": result["record"]["geometry_id"], "method": result["method"],
            "counts": counts, "reported_vacant": counts["vacant"],
            "vacant_range": [counts["vacant"], counts["vacant"] + counts["unknown"]],
            "warnings": result["warnings"],
            "geometry_quality": {k: v for k, v in result["quality"].items() if k != "slot_flags"},
            "geometry_quality_counts": result["geometry"]["quality_counts"],
            "line_segments": len(result["geometry"]["segments"]),
            "tiles": [{"tile_id": t["tile_id"], "bounds_xyxy": t["bounds_xyxy"],
                       "offset_xy": t["offset_xy"], "scale_xy": t["scale_xy"],
                       "padding_ltrb": t["padding_ltrb"]} for t in result["tiles"]],
            "detections": [asdict(d) for d in result["detections"]],
            "kept_detection_indices": result["kept_detection_indices"],
            "timing": result["timing"],
            "slots": [{**p, "space_id": p["slot_id"], "raw_state": raw[p["slot_id"]],
                       "boundary_support": geometry_slots[p["slot_id"]]["boundary_support"],
                       "geometry_quality_flag": geometry_slots[p["slot_id"]]["quality_flag"],
                       "polygon": polygons[p["slot_id"]]} for p in predictions],
        }


    def export_pipeline_results(results, output_dir, run_config, predictions_by_sample=None):
        """
        slots.csv, frames.jsonl, predictions.jsonl (đầu vào evaluate_occupancy),
        timings.jsonl (FrameTimer), warnings.txt, run_config.json.
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        predictions_by_sample = predictions_by_sample or {}
        with (output_dir / "slots.csv").open("w", newline="", encoding="utf-8") as slot_csv, \
                (output_dir / "frames.jsonl").open("w", encoding="utf-8") as frames_out, \
                (output_dir / "predictions.jsonl").open("w", encoding="utf-8") as predictions_out, \
                (output_dir / "timings.jsonl").open("w", encoding="utf-8") as timings_out, \
                (output_dir / "warnings.txt").open("w", encoding="utf-8") as warnings_out:
            writer = csv.DictWriter(slot_csv, fieldnames=SLOT_CSV_FIELDS, extrasaction="ignore")
            writer.writeheader()
            for result in results:
                predictions = predictions_by_sample.get(result["frame"].sample_id, result["predictions"])
                record = frame_record(result, predictions)
                frames_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                timings_out.write(json.dumps(result["timing"]) + "\n")
                for p in predictions:
                    predictions_out.write(json.dumps(
                        {"sample_id": p["sample_id"], "slot_id": p["slot_id"], "state": p["state"]}) + "\n")
                for slot in record["slots"]:
                    writer.writerow({**slot, "camera": record["camera"], "weather": record["weather"],
                                     "date": record["date"], "method": record["method"],
                                     "geometry_flags": "|".join(slot.get("geometry_flags", [])),
                                     "polygon": json.dumps(slot["polygon"])})
                for message in result["warnings"]:
                    warnings_out.write(message + "\n")
        (output_dir / "run_config.json").write_text(
            json.dumps(run_config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        return output_dir


    def pipeline_config(method, occupancy_config=None):
        config = {"method": method, "protocol": PROTOCOL, "geometry_config": GEOMETRY_CONFIG,
                  "geometry_quality_config": GEOMETRY_QUALITY_CONFIG,
                  "weak_geometry_policy": WEAK_GEOMETRY_POLICY,
                  "module_source": globals().get("provenance", {})}
        if method == "hog_svm":
            config["hog"] = {k: v for k, v in HOG_SVM_BUNDLE.items() if k != "model"}
        else:
            config["detector_config"] = dict(FULLFRAME_DETECTOR.config)
            config["occupancy_config"] = dict(OCCUPANCY_CONFIG if occupancy_config is None else occupancy_config)
            if method == "yolo_hybrid":
                config["hybrid_config"] = HYBRID_CONFIG
        return config


    # ---------- Trực quan hóa ----------
    def _draw_polygon(ax, polygon, color, lw=1.2, fill_alpha=0.0):
        polygon = np.asarray(polygon, dtype=float)
        closed = np.vstack([polygon, polygon[0]])
        if fill_alpha:
            ax.fill(polygon[:, 0], polygon[:, 1], color=color, alpha=fill_alpha, lw=0)
        ax.plot(closed[:, 0], closed[:, 1], color=color, lw=lw)


    def visualize_pipeline_result(result, predictions=None, save_path=None, show=True):
        """
        6 panel: (1) ảnh gốc, (2) biên Canny trong vùng bãi, (3) đoạn Hough + ô cờ yếu,
        (4) ROI/tile, (5) box xe trên ảnh gốc + phần dưới dùng để ghép, (6) ô tô màu theo trạng thái.
        Cần result tạo với keep_images=True.
        """
        if "image_bgr" not in result:
            raise ValueError("Gọi run_pipeline_frame(..., keep_images=True) để trực quan hóa.")
        predictions = predictions or result["predictions"]
        rgb = cv2.cvtColor(result["image_bgr"], cv2.COLOR_BGR2RGB)
        frame, layout, quality = result["frame"], result["layout"], result["quality"]
        weak = set(quality["weak_slot_ids"])
        counts = {s: sum(p["state"] == s for p in predictions) for s in STATE_COLORS}
        state_by_slot = {p["slot_id"]: p["state"] for p in predictions}
        fig, axes = plt.subplots(2, 3, figsize=(24, 10.5), layout="constrained")
        axes = axes.ravel()
        for ax in axes:
            ax.imshow(rgb)
            ax.axis("off")

        axes[0].set_title(f"1. Ảnh gốc — {frame.camera} / {frame.weather}")
        axes[1].clear()
        axes[1].imshow(result["geometry_images"]["roi_edges"], cmap="gray")
        axes[1].axis("off")
        axes[1].set_title("2. Canny trong vùng bãi (Gaussian → Canny → mask)")

        for space in layout["spaces"]:
            _draw_polygon(axes[2], space["polygon"], "orange" if str(space["space_id"]) in weak else "magenta",
                          lw=2 if str(space["space_id"]) in weak else 0.8)
        for segment in result["geometry"]["segments"]:
            x1, y1, x2, y2 = segment["xyxy"]
            axes[2].plot([x1, x2], [y1, y2], color="#32dc64" if segment["supports_boundary"] else "#00c8ff", lw=1.2)
        shift = quality["camera_shift"]
        axes[2].set_title(f"3. Hough: {len(result['geometry']['segments'])} đoạn | ô mất vạch (cam): {len(weak)}"
                          + (f" | lệch ({shift['dx']:+.1f}, {shift['dy']:+.1f}) px" if shift else " | không có tham chiếu"))

        if result["tiles"]:
            for tile in result["tiles"]:
                left, top, right, bottom = tile["bounds_xyxy"]
                axes[3].add_patch(plt.Rectangle((left, top), right - left, bottom - top, fill=False,
                                                edgecolor="cyan", linestyle="--", lw=1.5))
                axes[3].text(left + 4, top + 16, tile["tile_id"], color="black", fontsize=9,
                             bbox={"facecolor": "cyan", "alpha": 0.85, "pad": 1})
            axes[3].set_title(f"4. ROI/tile: {len(result['tiles'])} tile (mục 4.3)")
        else:
            roi = parking.make_parking_tiles(result["image_bgr"], layout, **TILE_CONFIG)["roi_bounds_xyxy"]
            axes[3].add_patch(plt.Rectangle((roi[0], roi[1]), roi[2] - roi[0], roi[3] - roi[1],
                                            fill=False, edgecolor="yellow", lw=2))
            axes[3].set_title("4. ROI bãi xe (phương pháp này không chia tile)")

        lower = (result["occupancy_config"] or {}).get("lower_fraction", 0.5)
        for index in result["kept_detection_indices"]:
            d = result["detections"][index]
            x1, y1, x2, y2 = d.xyxy
            axes[4].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="deepskyblue", lw=1.2))
            axes[4].add_patch(plt.Rectangle((x1, y2 - (y2 - y1) * lower), x2 - x1, (y2 - y1) * lower,
                                            fill=False, edgecolor="yellow", lw=0.8))
            axes[4].text(x1, y1 - 2, f"{d.class_name} {d.confidence:.2f}", color="deepskyblue", fontsize=6)
        axes[4].set_title(f"5. Box xe trên ảnh gốc: {len(result['kept_detection_indices'])} (vàng = phần dưới để ghép)"
                          if result["method"] != "hog_svm" else "5. HOG/SVM phân loại từng ô: không có box xe")

        for space in layout["spaces"]:
            _draw_polygon(axes[5], space["polygon"], STATE_COLORS[state_by_slot[str(space["space_id"])]],
                          lw=1.5, fill_alpha=0.35)
        axes[5].set_title(f"6. {METHOD_LABELS[result['method']]}: trống {counts['vacant']} | "
                          f"có xe {counts['occupied']} | chưa xác định {counts['unknown']}")
        axes[5].legend(handles=[Line2D([0], [0], color=c, lw=4, label=s) for s, c in STATE_COLORS.items()],
                       loc="lower right", fontsize=9)
        flags = quality["frame_flags"]
        fig.suptitle(f"{frame.sample_id} — {result['timing']['total_ms']:.0f} ms/ảnh, "
                     f"{result['timing']['inference_calls']} lượt suy luận"
                     + (f" — CỜ HÌNH HỌC YẾU: {', '.join(flags)}" if flags else ""),
                     fontsize=13, color="darkred" if flags else "black")
        if save_path is not None:
            save_figure(fig, save_path, dpi=80)
        if show:
            plt.show()
        plt.close(fig)


    def print_frame_report(result):
        c, t = result["counts"], result["timing"]
        print(f"[{METHOD_LABELS[result['method']]}] {result['frame'].sample_id}")
        print(f"  Trống {c['vacant']} | Có xe {c['occupied']} | Chưa xác định {c['unknown']} "
              f"-> số chỗ trống báo cáo: {c['vacant']} (khoảng {c['vacant']}–{c['vacant'] + c['unknown']})")
        print(f"  {t['total_ms']:.1f} ms (pre {t['preprocess_ms']:.1f} | model {t['model_ms']:.1f} | "
              f"post {t['postprocess_ms']:.1f}), {t['inference_calls']} lượt suy luận")
        for message in result["warnings"]:
            print("  ⚠", message)


    # ---------- Chạy thử end-to-end: một ảnh validation, cả ba phương pháp ----------
    # None: ảnh validation đầu tiên có cả ô trống và ô có xe theo nhãn (chỉ để minh họa).
    PIPELINE_DEMO_INDEX = None
    if PIPELINE_DEMO_INDEX is None:
        PIPELINE_DEMO_INDEX = next((i for i, r in enumerate(VAL_FRAMES)
                                    if 0 < sum(r["labels"].values()) < len(r["labels"])), 0)
    demo_record = VAL_FRAMES[PIPELINE_DEMO_INDEX]
    demo_dir = PIPELINE_OUTPUT_DIR / "demo"
    FULLFRAME_DETECTOR.warmup()
    for method in PIPELINE_METHODS:
        demo_result = run_pipeline_frame(demo_record, method, keep_images=True)
        print_frame_report(demo_result)
        visualize_pipeline_result(demo_result, save_path=demo_dir / f"{method}.png")
        retry_io(lambda: export_pipeline_results([demo_result], demo_dir / method, pipeline_config(method)))
    truth = demo_record["labels"]
    print("Nhãn thật (chỉ để đối chiếu):", sum(v == 0 for v in truth.values()), "trống /",
          sum(truth.values()), "có xe. Cấu hình ghép đang là bản xem trước của 4.6; 6.2 chốt lại.")
    print("Hình và CSV/JSON demo:", demo_dir)

    globals().get("sync_outputs_to_drive", lambda: None)()

#### Định dạng đầu ra của pipeline

Mỗi lần chạy ghi vào `OUTPUT_DIR/<PROTOCOL>/pipeline/<split>/<method>/`:

| File | Nội dung |
|---|---|
| `slots.csv` | Mỗi dòng một ô: `sample_id, space_id, camera, weather, date, method, state (vacant/occupied/unknown), raw_state (trước chính sách hình học), score, match_score, detection_index, geometry_flags, boundary_support, geometry_quality_flag, polygon`. |
| `frames.jsonl` | Mỗi dòng một ảnh: số ô trống/có xe/chưa xác định, `reported_vacant`, `vacant_range`, cảnh báo, chất lượng hình học (cờ, độ lệch camera), số đoạn thẳng, tile (offset/tỉ lệ/padding), box xe trên ảnh gốc, box được giữ, timing, danh sách ô. |
| `predictions.jsonl` | Đầu vào `parking.evaluate_occupancy`: `sample_id, slot_id, state`. |
| `timings.jsonl` | Một dòng/ảnh gốc: `preprocess_ms, model_ms, postprocess_ms, total_ms, inference_calls`. |
| `warnings.txt` | Toàn bộ thông báo hình học yếu. |
| `run_config.json` | Cấu hình hình học, chất lượng hình học, chính sách, detector/hybrid/ghép hoặc HOG, nguồn module. |
| `evaluation/` | `metrics.json`, `per_frame.csv`, `per_slot.csv`, `aligned_predictions.jsonl`. |

Số chỗ trống báo cáo chỉ đếm ô `vacant`. `vacant_range` là khoảng giá trị nếu tính thêm các ô `unknown`.


## 6. Đánh giá và thực nghiệm


### 6.1. Chỉ số và bộ đánh giá chung

Ghép theo `(sample_id, space_id)`; có xe là lớp dương. Ô thiếu/chưa xác định làm giảm coverage và tính sai trong accuracy tổng. Recall/F1 chính phạt ô có xe chưa xác định; `covered_only` chỉ dùng để chẩn đoán. Báo tỉ lệ báo trống sai, MAE đếm ô trống và thời gian trung bình trên ảnh gốc (gồm mọi tile).

Cell dưới kiểm chứng bằng ví dụ nhân tạo tính tay, không phải kết quả mô hình. Khi có dự đoán, điền đường dẫn JSONL; ngưỡng chỉ chọn trên validation rồi giữ nguyên khi test. Xem `docs/evaluation.md` trong repo để biết schema và công thức.


In [ ]:
# Kiểm chứng độc lập, không dùng làm kết quả thực nghiệm.
print("Evaluator self-check (synthetic):")
print(json.dumps(parking.evaluator_self_check(), indent=2))

# Điền sau khi pipeline đã xuất dự đoán cho toàn bộ split tương ứng.
EVAL_SPLIT = "val"  # Chỉ chuyển sang test sau khi đã khóa tham số.
METHOD_NAME = "method_name"
PREDICTIONS_PATH = ""  # JSONL: sample_id, space_id, occupied_score hoặc state.
TIMINGS_PATH = ""      # JSONL: thời gian trên từng ảnh gốc; để trống nếu chưa đo.
EVALUATION_THRESHOLD = 0.5

if PREDICTIONS_PATH:
    eval_frames = project_data["protocols"][PROTOCOL][EVAL_SPLIT]
    eval_metadata = dict(
        dataset_version=PROJECT_DATASET_VERSION, protocol=PROTOCOL, split=EVAL_SPLIT,
        split_lock_sha256=hashlib.sha256((PROJECT_SPLIT_DIR / "split_lock.json").read_bytes()).hexdigest(),
        predictions_sha256=hashlib.sha256(Path(PREDICTIONS_PATH).read_bytes()).hexdigest(),
        module_sha256=provenance["sha256"],
    )
    if TIMINGS_PATH:
        eval_metadata["timings_sha256"] = hashlib.sha256(Path(TIMINGS_PATH).read_bytes()).hexdigest()
    evaluation_report = parking.evaluate_occupancy(
        eval_frames, PREDICTIONS_PATH, timings=TIMINGS_PATH or None,
        threshold=EVALUATION_THRESHOLD, missing_policy="unknown",
        method=METHOD_NAME, metadata=eval_metadata,
    )
    evaluation_dir = Path(OUTPUT_DIR) / PROTOCOL / "evaluation" / METHOD_NAME / EVAL_SPLIT
    parking.export_evaluation(evaluation_report, evaluation_dir)
    print(json.dumps(evaluation_report["overall"], indent=2))
    print("Saved:", evaluation_dir)
else:
    print("Bộ đánh giá đã qua kiểm tra nhân tạo. Kết quả thật được đánh giá ở mục 6.2 (validation) và 6.3 (test).")


### 6.2. Validation và cấu hình đã khóa

Các cell demo cũ bên dưới giữ để tham khảo khi tắt runner chung. Luồng chính là cell **Dùng lại validation đã khóa hoặc chạy validation mới**: có artifact rồi thì không tìm kiếm lại, chạy tiếp 6.3 trên cùng notebook.


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # Chỉ dùng validation. Hai phương pháp dùng chung một cấu hình đã chọn.
    # Xếp hạng: mean F1 (unknown bị phạt), rồi coverage, rồi MAE số ô trống.
    # Không chạy lại YOLO cho từng cấu hình; dùng box đã có từ 4.4 và 4.5.
    OCCUPANCY_CANDIDATES = [
        {**parking.DEFAULT_OCCUPANCY_CONFIG, "lower_fraction": fraction,
         "min_overlap": overlap, "ambiguity_margin": margin}
        for fraction in (.33, .5, .67)
        for overlap in (.2, .3, .4)
        for margin in (0., .05)
    ]

    occupancy_search = []
    best_key = None
    best_results = None
    best_reports = None
    best_config = None
    for candidate_index, candidate in enumerate(OCCUPANCY_CANDIDATES):
        candidate_results = run_occupancy(candidate)
        reports = {}
        for method, frames in candidate_results.items():
            predictions = [p for result in frames.values() for p in result["predictions"]]
            reports[method] = parking.evaluate_occupancy(
                VAL_FRAMES, predictions, missing_policy="error", method=method,
                metadata={"split": "val", "protocol": PROTOCOL, "occupancy_config": candidate},
            )
        metrics = {method: report["overall"] for method, report in reports.items()}
        mean_f1 = sum(m["f1"] for m in metrics.values()) / len(metrics)
        mean_coverage = sum(m["coverage"] for m in metrics.values()) / len(metrics)
        mean_mae = sum(m["mae_vacant_count"] for m in metrics.values()) / len(metrics)
        rank = (mean_f1, mean_coverage, -mean_mae)
        occupancy_search.append({"config": candidate, "mean_f1": mean_f1,
                                 "mean_coverage": mean_coverage, "mean_mae_vacant_count": mean_mae,
                                 "methods": metrics})
        if best_key is None or rank > best_key:
            best_key = rank
            best_config = dict(candidate)
            best_results = candidate_results
            best_reports = reports
        print(f"Matching validation {candidate_index+1}/{len(OCCUPANCY_CANDIDATES)}: mean F1={mean_f1:.4f}")

    FROZEN_OCCUPANCY_CONFIG = dict(best_config)
    OCCUPANCY_VAL = best_results
    occupancy_validation_dir = OCCUPANCY_OUTPUT_DIR / "validation"
    export_occupancy(OCCUPANCY_VAL, occupancy_validation_dir)
    for method, report in best_reports.items():
        parking.export_evaluation(report, occupancy_validation_dir / method / "evaluation")
        metrics = report["overall"]
        print(method, {key: metrics[key] for key in
                       ("f1", "coverage", "false_vacancy_rate", "mae_vacant_count")})

    # Lưu tham số và nguồn dữ liệu để tái sử dụng nguyên trạng khi chạy test.
    # F1 ở đây dùng để chọn cấu hình, không phải kết quả test độc lập.
    detection_fingerprints = {}
    for method, frames in OCCUPANCY_DETECTIONS.items():
        serialized = {sample_id: [{"xyxy": list(d.xyxy), "confidence": d.confidence,
                                  "class_name": d.class_name} for d in detections]
                      for sample_id, detections in frames.items()}
        detection_fingerprints[method] = hashlib.sha256(
            json.dumps(serialized, sort_keys=True).encode()
        ).hexdigest()
    frozen_matching = {
        "config": FROZEN_OCCUPANCY_CONFIG,
        "dataset_version": PROJECT_DATASET_VERSION, "protocol": PROTOCOL, "selected_on": "val",
        "selection_rule": "mean F1 across both methods; ties: higher coverage, lower vacant-count MAE, grid order",
        "val_manifest_sha256": hashlib.sha256((MANIFEST_DIR / "val.jsonl").read_bytes()).hexdigest(),
        "module_sha256": provenance["sha256"],
        "detection_sha256": detection_fingerprints,
        "detector_config": dict(FULLFRAME_DETECTOR.config), "hybrid_config": HYBRID_CONFIG,
    }
    (occupancy_validation_dir / "frozen_config.json").write_text(
        json.dumps(frozen_matching, indent=2), encoding="utf-8"
    )
    (occupancy_validation_dir / "search.json").write_text(
        json.dumps(occupancy_search, indent=2), encoding="utf-8"
    )
    print("Shared configuration:", FROZEN_OCCUPANCY_CONFIG)
    print("Validation outputs:", occupancy_validation_dir)
    plot_occupancy(OCCUPANCY_VAL, VAL_FRAMES[0])


#### Tham khảo luồng demo cũ: cờ hình học yếu và khóa pipeline

Cell này dùng lại dự đoán validation sẵn có, nên không chạy lại YOLO:
- `OCCUPANCY_VAL` với cấu hình ghép chung vừa chọn ở trên.
- `HOG_VAL_PREDICTIONS` của 4.1, ngưỡng margin 0.

Cell tính cờ hình học cho từng ảnh validation rồi so sánh chính sách `unknown` (bắt buộc theo yêu cầu) với `report_only` (giữ nguyên trạng thái, chỉ để đo tác động). Chính sách không được chọn theo điểm số. Bảng cho biết tỉ lệ báo trống sai giảm bao nhiêu và coverage giảm bao nhiêu. Kết quả được ghi vào `pipeline/locked_config.json` cùng SHA256 manifest validation, để mục 6.3 dùng nguyên trạng.


In [ ]:
if globals().get("VALIDATION_ONLY_MODE", False):
    print("Dùng runner chung ở mục 6.2 (validation) và 6.3 (test); bỏ demo này để tránh chạy trùng.")
else:
    # ============================================================
    # 6.2b. Cờ hình học yếu trên validation và khóa cấu hình pipeline
    # - Dùng lại dự đoán validation đã có: OCCUPANCY_VAL (cấu hình chung đã chọn ở trên)
    #   và HOG_VAL_PREDICTIONS (4.1). Không chạy lại YOLO.
    # - Tính cờ hình học cho từng ảnh validation (4.2 + tham chiếu train ở mục 5).
    # - So sánh chính sách "unknown" (bắt buộc) với "report_only" (chỉ để phân tích).
    # - Ghi pipeline/locked_config.json cho mục 6.3. Không dùng test.
    # ============================================================

    import hashlib
    import importlib.util
    import json
    import subprocess
    import sys

    if importlib.util.find_spec("pandas") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "pandas"], check=True)
    import pandas as pd

    for name in ("FROZEN_OCCUPANCY_CONFIG", "OCCUPANCY_VAL", "HOG_VAL_PREDICTIONS", "GEOMETRY_REFERENCES"):
        if name not in globals():
            raise RuntimeError(f"Thiếu {name}: chạy 4.1, mục 5 và cell 6.2 phía trên trước.")
    REPORT_KEYS = ["f1", "accuracy", "coverage", "precision", "recall",
                   "false_vacancy_rate", "reported_vacancy_error_rate", "mae_vacant_count", "unknown_slots"]

    # ---------- 1. Cờ hình học trên từng ảnh validation ----------
    VAL_GEOMETRY_QUALITY, geometry_rows = {}, []
    started = time.perf_counter()
    for record in VAL_FRAMES:
        frame = parking.read_project_frame(record, DATASET_DIR)
        geometry = parking.analyze_parking_geometry(read_bgr(frame), SLOT_GEOMETRIES[record["geometry_id"]],
                                                    GEOMETRY_CONFIG)
        quality = parking.assess_geometry_quality(
            geometry, GEOMETRY_REFERENCES.get(record["geometry_id"]), GEOMETRY_QUALITY_CONFIG)
        VAL_GEOMETRY_QUALITY[record["sample_id"]] = quality
        geometry_rows.append({
            "sample_id": record["sample_id"], "camera": record["camera"], "weather": record["weather"],
            "frame_weak": bool(quality["frame_flags"]), "frame_flags": "|".join(quality["frame_flags"]),
            "weak_slots": len(quality["weak_slot_ids"]), "slots": len(geometry["slots"]),
            "segments": len(geometry["segments"]),
            "supported_fraction": geometry["quality_counts"]["supported"] / len(geometry["slots"]),
            "shift_px": None if quality["camera_shift"] is None
            else float(np.hypot(quality["camera_shift"]["dx"], quality["camera_shift"]["dy"])),
        })
    print(f"Hình học validation: {len(VAL_FRAMES)} ảnh, {time.perf_counter() - started:.1f}s")
    val_geometry_df = pd.DataFrame(geometry_rows)

    # ---------- 2. Dự đoán validation trước chính sách ----------
    VAL_RAW_PREDICTIONS = {"hog_svm": {}}
    hog_threshold = HOG_SVM_BUNDLE.get("decision_threshold", 0.0)
    for p in HOG_VAL_PREDICTIONS:
        VAL_RAW_PREDICTIONS["hog_svm"].setdefault(p.sample_id, []).append({
            "sample_id": p.sample_id, "slot_id": p.space_id, "score": p.occupied_score,
            "state": "occupied" if p.occupied_score > hog_threshold else "vacant"})
    for method in ("yolo_fullframe", "yolo_hybrid"):
        VAL_RAW_PREDICTIONS[method] = {sid: r["predictions"] for sid, r in OCCUPANCY_VAL[method].items()}

    # ---------- 3. Hai chính sách, ba phương pháp ----------
    VAL_REPORTS, VAL_FINAL_PREDICTIONS, policy_rows = {}, {}, []
    VALIDATION_DIR = PIPELINE_OUTPUT_DIR / "val"
    for method in PIPELINE_METHODS:
        for policy in ("unknown", "report_only"):
            by_sample = {sid: parking.apply_geometry_policy(rows, VAL_GEOMETRY_QUALITY[sid], policy)["predictions"]
                         for sid, rows in VAL_RAW_PREDICTIONS[method].items()}
            report = parking.evaluate_occupancy(
                VAL_FRAMES, [p for rows in by_sample.values() for p in rows], missing_policy="error",
                method=method, metadata={"split": "val", "protocol": PROTOCOL, "weak_geometry_policy": policy})
            policy_rows.append({"method": METHOD_LABELS[method], "policy": policy,
                                **{k: report["overall"][k] for k in REPORT_KEYS}})
            if policy == WEAK_GEOMETRY_POLICY:
                VAL_REPORTS[method], VAL_FINAL_PREDICTIONS[method] = report, by_sample
                out = VALIDATION_DIR / method
                out.mkdir(parents=True, exist_ok=True)
                (out / "predictions.jsonl").write_text("".join(
                    json.dumps({"sample_id": p["sample_id"], "slot_id": p["slot_id"], "state": p["state"],
                                "geometry_flags": p["geometry_flags"]}) + "\n"
                    for rows in by_sample.values() for p in rows), encoding="utf-8")
                parking.export_evaluation(report, out / "evaluation")
    policy_df = pd.DataFrame(policy_rows)
    policy_df.to_csv(VALIDATION_DIR / "weak_geometry_policy.csv", index=False)
    val_geometry_df.to_csv(VALIDATION_DIR / "geometry_flags.csv", index=False)

    # ---------- 4. Khóa cấu hình pipeline ----------
    LOCKED_PIPELINE_CONFIG = {
        "dataset_version": PROJECT_DATASET_VERSION, "protocol": PROTOCOL, "selected_on": "val",
        "val_manifest_sha256": hashlib.sha256((MANIFEST_DIR / "val.jsonl").read_bytes()).hexdigest(),
        "occupancy_config": FROZEN_OCCUPANCY_CONFIG,
        "hog_decision_threshold": hog_threshold, "hog_selected_C": HOG_SVM_BUNDLE.get("selected_C"),
        "geometry_config": GEOMETRY_CONFIG, "geometry_quality_config": GEOMETRY_QUALITY_CONFIG,
        "geometry_reference_frames": GEOMETRY_REFERENCE_FRAMES,
        "weak_geometry_policy": WEAK_GEOMETRY_POLICY,
        "detector_config": dict(FULLFRAME_DETECTOR.config), "hybrid_config": HYBRID_CONFIG,
        "module_source": globals().get("provenance", {}),
    }
    (PIPELINE_OUTPUT_DIR / "locked_config.json").write_text(
        json.dumps(LOCKED_PIPELINE_CONFIG, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    print("\nChính sách hình học yếu trên validation:")
    display(policy_df.round(4))
    print("\nCờ hình học trên validation theo camera/thời tiết:")
    display(val_geometry_df.groupby(["camera", "weather"]).agg(
        frames=("sample_id", "count"), frame_weak_rate=("frame_weak", "mean"),
        weak_slots_mean=("weak_slots", "mean"), supported_fraction=("supported_fraction", "median"),
        segments=("segments", "median"), shift_px_median=("shift_px", "median")).round(3))
    print("Đã khóa cấu hình pipeline:", PIPELINE_OUTPUT_DIR / "locked_config.json")

    globals().get("sync_outputs_to_drive", lambda: None)()

#### Luồng chính: dùng lại validation đã khóa hoặc chạy validation mới

**Có kết quả rồi:** giữ `RUN_VALIDATION_SEARCH=False`. Cell dưới chỉ tìm artifact và đọc summary/khóa, không huấn luyện hay inference lại. Sau đó đi tiếp mục 6.3 trên cùng notebook.

**Chưa có kết quả:** bật `RUN_VALIDATION_SEARCH=True` ở mục 2 để chạy dải đã đăng ký. Chỉ run đầy đủ mới tạo khóa. Cache không tương thích phải dùng thư mục run mới, không ép dùng lại.

Ưu tiên artifact trong bundle, `WORK_DIR/outputs/validation_v1`, hoặc thư mục output của Colab. Có thể chỉ định chính xác qua `VALIDATION_ARTIFACT_DIR`. Khóa, model HOG và tham chiếu train phải cùng lượt; không chỉ lấy mỗi locked_config.json.


In [ ]:
# Cấu hình và đọc lại artifact trước; không tự chạy tìm kiếm khi đã có khóa.
artifact_candidates = [Path(globals().get("LOCKED_BUNDLE_ROOT", WORK_DIR / "outputs/locked_bundle")) / "outputs" / TASK12_RUN_NAME,
                       WORK_DIR / "outputs" / TASK12_RUN_NAME,
                       Path(OUTPUT_DIR) / PROTOCOL / TASK12_RUN_NAME]
if globals().get("VALIDATION_ARTIFACT_DIR", ""):
    TASK12_DIR = Path(VALIDATION_ARTIFACT_DIR).expanduser()
else:
    TASK12_DIR = next((path for path in artifact_candidates if (path / "locked_config.json").is_file()),
                      Path(OUTPUT_DIR) / PROTOCOL / TASK12_RUN_NAME)
TASK12_SUMMARY = None
registration = TASK12_DIR / "preregistered_plan.json"
if registration.is_file():
    VALIDATION_PLAN = json.loads(registration.read_text())["plan"]
else:
    plan_candidates = [Path(globals().get("LOCKED_BUNDLE_ROOT", WORK_DIR / "outputs/locked_bundle")) / "configs/validation_v1.json",
                       WORK_DIR / "configs/validation_v1.json"]
    plan_file = next((path for path in plan_candidates if path.is_file()), None)
    if plan_file is not None:
        VALIDATION_PLAN = json.loads(plan_file.read_text())
    elif globals().get("RUN_VALIDATION_SEARCH", False):
        plan_url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/{quote(REPO, safe='')}/"
                    f"{quote(REF, safe='')}/configs/validation_v1.json?refresh={time.time_ns()}")
        with urllib.request.urlopen(plan_url, timeout=60) as response:
            VALIDATION_PLAN = json.loads(response.read())
    else:
        VALIDATION_PLAN = None
if VALIDATION_PLAN is not None and VALIDATION_PLAN["protocol"] != PROTOCOL:
    raise ValueError("Plan/khóa hiện tại dành cho giao thức khác. Không trộn kết quả giữa các protocol.")
checkpoint_name = VALIDATION_PLAN["checkpoint"]["filename"] if VALIDATION_PLAN is not None else "yolo11n.pt"
checkpoint_candidates = [Path(globals().get("LOCKED_BUNDLE_ROOT", WORK_DIR / "outputs/locked_bundle")) / "models" / checkpoint_name,
                         WORK_DIR / "models" / checkpoint_name]
TASK12_CHECKPOINT = next((path for path in checkpoint_candidates if path.is_file()), WORK_DIR / "models" / checkpoint_name)
if globals().get("RUN_VALIDATION_SEARCH", False):
    if VALIDATION_PLAN is None:
        raise RuntimeError("Chưa có validation plan.")
    TASK12_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    if not TASK12_CHECKPOINT.is_file():
        temporary_checkpoint = TASK12_CHECKPOINT.with_suffix(".part")
        with urllib.request.urlopen(VALIDATION_PLAN["checkpoint"]["url"], timeout=120) as source:
            temporary_checkpoint.write_bytes(source.read())
        if parking._validation_sha(temporary_checkpoint) != VALIDATION_PLAN["checkpoint"]["sha256"]:
            raise ValueError("Checkpoint tải về không khớp SHA256.")
        temporary_checkpoint.replace(TASK12_CHECKPOINT)
    TASK12_SUMMARY = parking.run_validation_search(
        DATASET_DIR, MANIFEST_DIR, TASK12_DIR, VALIDATION_PLAN, TASK12_CHECKPOINT,
        device=globals().get("RUN_DEVICE", "auto"),
    )
    globals().get("sync_outputs_to_drive", lambda: None)()
else:
    summary_file = TASK12_DIR / "validation_summary.json"
    if summary_file.is_file():
        TASK12_SUMMARY = json.loads(summary_file.read_text())
    print("Không chạy lại validation. Dùng artifact đã có:", TASK12_DIR)
if (TASK12_DIR / "locked_config.json").is_file():
    saved_lock = json.loads((TASK12_DIR / "locked_config.json").read_text())
    print("Khóa tìm thấy; selected_on:", saved_lock.get("selected_on"), "complete:", saved_lock.get("complete"))
    print("Bước 6.3a sẽ kiểm tra hash/model/môi trường và dataset.")
else:
    print("Chưa có khóa: khôi phục bundle/VALIDATION_ARTIFACT_DIR, hoặc chạy validation mới. Chưa chạy test.")
print("Checkpoint:", TASK12_CHECKPOINT)


### 6.3. Chạy test và lưu bằng chứng trên cùng notebook

Không cần notebook thứ hai. Luồng: **6.3a kiểm tra → 6.3b chạy test → 6.3c bảng → 6.3d ảnh → 6.3e checksum**. Giữ nguyên model/split/ngưỡng đã chọn trên validation. Các bước xuất chỉ đọc dự đoán đã lưu, không inference lại.

Cần `TASK12_DIR`, `TASK12_CHECKPOINT` từ cell 6.2 ngay trên; cell đó không chạy lại validation khi cờ False. `RUN_FINAL_TEST` mặc định False; tự bật ở cấu hình khi sẵn sàng. Runtime mới cần lấy model HOG và tham chiếu từ Drive/bundle.

Notebook cũ từng có test output; lượt mới không chọn ngưỡng từ test nhưng không gọi split này là blind holdout chưa từng quan sát.


#### 6.3a. Kiểm tra khóa, model và dữ liệu

Chỉ kiểm tra artifact/XML; chưa chạy model trên test. Nếu báo lệch thư viện/hash thì sửa đúng đầu vào, không tắt kiểm tra.


In [ ]:
# 6.3a. Import helper từ repo/bundle; fallback GitHub nếu chưa có.
helper_candidates = [Path(globals().get("LOCKED_BUNDLE_ROOT", WORK_DIR / "outputs/locked_bundle")) / "final_test_module.py",
                     WORK_DIR / "final_test_module.py"]
helper_file = next((path for path in helper_candidates if path.is_file()), None)
if helper_file is None:
    helper_url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/{quote(REPO, safe='')}/"
                  f"{quote(REF, safe='')}/final_test_module.py?refresh={time.time_ns()}")
    helper_file = WORK_DIR / "outputs/runtime/final_test_module.py"
    helper_file.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(helper_url, timeout=60) as response:
            helper_file.write_bytes(response.read())
    except Exception as error:
        raise RuntimeError("Thiếu final_test_module.py. Dùng bundle hoặc đặt helper cạnh parking_module.py trong WORK_DIR.") from error
helper_spec = importlib.util.spec_from_file_location("final_test_module", helper_file)
final_test = importlib.util.module_from_spec(helper_spec)
sys.modules[helper_spec.name] = final_test
helper_spec.loader.exec_module(final_test)
FINAL_TEST_DIR = (Path(FINAL_TEST_OUTPUT_DIR).expanduser() if globals().get("FINAL_TEST_OUTPUT_DIR", "")
                  else Path(OUTPUT_DIR) / PROTOCOL / globals().get("FINAL_TEST_RUN_NAME", "locked_test_v1"))
FINAL_TEST_INPUTS = None
required_artifacts = [TASK12_DIR / name for name in ("locked_config.json", "preregistered_plan.json", "hog_svm.joblib", "geometry_references.joblib")]
required_artifacts.append(TASK12_CHECKPOINT)
missing_artifacts = [str(path) for path in required_artifacts if not path.is_file()]
if missing_artifacts:
    print("Thiếu artifact đã khóa; khôi phục bundle hoặc sửa VALIDATION_ARTIFACT_DIR:", *missing_artifacts, sep="\n")
    if globals().get("RUN_FINAL_TEST", False):
        raise FileNotFoundError("Chưa đủ artifact, không được bắt đầu test.")
else:
    FINAL_TEST_INPUTS = final_test.check_inputs(DATASET_DIR, MANIFEST_DIR, TASK12_DIR, TASK12_CHECKPOINT)
    print(json.dumps(FINAL_TEST_INPUTS, indent=2))
print("Output test:", FINAL_TEST_DIR)


#### 6.3b. Chạy test thật hoặc đọc kết quả đã có

Chạy đủ bốn phương pháp; output mới/rỗng, không ghi đè. Nếu chỉ muốn đọc lượt đã hoàn tất, giữ RUN_FINAL_TEST=False. Nếu lần trước bị ngắt, giữ bằng chứng lỗi và chọn tên run mới.


In [ ]:
# 6.3b. Cell duy nhất chạy model trên test trong luồng chính thức.
TEST_REPORTS = {}
if globals().get("RUN_FINAL_TEST", False):
    if not globals().get("FINAL_TEST_INPUTS"):
        raise RuntimeError("Chạy thành công cell 6.3a trước.")
    TEST_REPORTS = final_test.run_final_test(
        DATASET_DIR, MANIFEST_DIR, TASK12_DIR, TASK12_CHECKPOINT,
        FINAL_TEST_DIR, device=globals().get("RUN_DEVICE", "auto"),
    )
    globals().get("sync_outputs_to_drive", lambda: None)()
elif globals().get("FINAL_TEST_DIR") and (FINAL_TEST_DIR / "test_summary.json").is_file():
    TEST_REPORTS = final_test.verify_saved_test(FINAL_TEST_DIR, MANIFEST_DIR, TASK12_DIR)
    print("Đọc test đã có, không chạy model lại:", FINAL_TEST_DIR)
else:
    print("Chưa chạy test. Khi sẵn sàng, bật RUN_FINAL_TEST=True ở cấu hình rồi chạy 6.3b.")


#### 6.3c. Bảng tổng, camera/thời tiết và timing

Xuất từ dự đoán đã lưu, kiểm tra lại metrics và pixel hybrid/highres từng ảnh. Có comparison, by_camera, by_weather, by_camera_weather, confusion_matrices, timings_per_frame, predictions/timings JSONL. Model stage bao gồm Ultralytics internal preprocess/NMS và mapping hybrid, không chỉ neural forward.


In [ ]:
FINAL_TEST_ROWS = []
if TEST_REPORTS:
    FINAL_TEST_ROWS = final_test.export_tables(FINAL_TEST_DIR, MANIFEST_DIR, TASK12_DIR)
    print("Bảng test:", FINAL_TEST_DIR / "tables")
else:
    print("Chưa có test hoàn chỉnh; chưa xuất bảng test.")


#### 6.3d. Ảnh đúng, sai và unknown

Mỗi phương pháp/camera/thời tiết chọn mẫu đầu theo sample_id: correct là mọi ô đều đúng/xác định; error có ít nhất một dự đoán xác định sai; unknown_only có từ chối nhưng không có lỗi xác định. Nhóm thiếu mẫu được ghi available=0. Không dùng ảnh lỗi để đổi tham số.

Overlay polygon: xanh trống, đỏ có xe, cam unknown; T nhãn thật, P dự đoán. Runner chưa lưu raw box vào báo cáo nên ảnh này là bằng chứng trạng thái ô, không phải raw detection. Quy tắc chọn và ảnh nguồn nằm trong selection.json. Ảnh đã có được dùng lại, không tạo lại khi đọc kết quả.


In [ ]:
FINAL_TEST_EXAMPLES = []
if TEST_REPORTS:
    selection_path = FINAL_TEST_DIR / "examples/selection.json"
    if selection_path.is_file():
        FINAL_TEST_EXAMPLES = json.loads(selection_path.read_text())["selected"]
        print("Dùng ảnh minh họa đã lưu:", len(FINAL_TEST_EXAMPLES))
    else:
        FINAL_TEST_EXAMPLES = final_test.export_examples(
            FINAL_TEST_DIR, MANIFEST_DIR, TASK12_DIR, DATASET_DIR,
            per_category=globals().get("TEST_EXAMPLES_PER_CATEGORY", 1),
        )
        print("Đã tạo ảnh minh họa:", len(FINAL_TEST_EXAMPLES))
    print("Ảnh và quy tắc chọn:", FINAL_TEST_DIR / "examples")
else:
    print("Chưa có test hoàn chỉnh; chưa tạo ảnh minh họa test.")


#### 6.3e. Hash bằng chứng và lưu Drive

Giữ toàn bộ thư mục test; hash kiểm tra tính toàn vẹn, không phải chứng nhận khoa học.


In [ ]:
if TEST_REPORTS:
    workflow_status = FINAL_TEST_DIR / "workflow_status.json"
    if workflow_status.is_file() and json.loads(workflow_status.read_text()).get("status") == "complete":
        print("Hash artifact:", final_test.seal_evidence(FINAL_TEST_DIR))
    else:
        print("Lượt test cũ không có workflow_status complete; giữ nguyên evidence, chưa đóng dấu workflow mới.")
    globals().get("sync_outputs_to_drive", lambda: None)()
else:
    print("Chưa có test để đóng dấu artifact.")


## 7. Kết quả và thảo luận


Kết quả đọc từ artifact đã kiểm tra, tách rõ validation (6.2) và test (6.3). Mỗi bên có F1, precision/recall, accuracy tổng, coverage, hai tỉ lệ báo trống sai, MAE đếm và timing toàn ảnh khi đã đo.

Validation chính thức đã có timing pipeline; không nhầm với bản demo cũ chỉ dùng dự đoán cache. Test có bảng theo camera/thời tiết/camera×thời tiết, ma trận 2×3 và timing từng ảnh. Bảng nằm trong `FINAL_TEST_DIR/tables`, ảnh cùng quy tắc chọn trong `FINAL_TEST_DIR/examples`.

Mục này chỉ hiển thị số đã lưu, không gọi model và không chọn tham số từ test. Khi chưa có lượt test mới, notebook nói rõ chưa có thay vì điền bằng số validation hoặc số test lịch sử.


In [ ]:
# Bảng validation và test luôn ghi rõ split; không phụ thuộc các demo 4–5.
if globals().get("TASK12_SUMMARY"):
    print("VALIDATION — dùng chọn tham số, không phải test độc lập:")
    validation_rows = [dict(split="val", method=method,
                           **{key: values[key] for key in ("precision", "recall", "f1", "accuracy", "coverage", "false_vacancy_rate", "mae_vacant_count")},
                           ms_per_image=values["timing"]["total_ms_mean"])
                       for method, values in TASK12_SUMMARY["methods"].items()]
    try:
        import pandas as pd
        from IPython.display import display
    except ImportError:
        print(json.dumps(validation_rows, ensure_ascii=False, indent=2))
    else:
        display(pd.DataFrame(validation_rows))
    print("Toàn bộ grid, kể cả không cải thiện:", TASK12_DIR)
if globals().get("TEST_REPORTS"):
    import csv
    from IPython.display import display
    print("TEST — cấu hình đã khóa từ validation:")
    for filename in ("comparison.csv", "by_camera.csv", "by_weather.csv", "by_camera_weather.csv"):
        path = FINAL_TEST_DIR / "tables" / filename
        if path.is_file():
            with path.open(encoding="utf-8", newline="") as stream:
                rows = list(csv.DictReader(stream))
            print(filename, "→", path)
            try:
                import pandas as pd
            except ImportError:
                print(json.dumps(rows, ensure_ascii=False, indent=2))
            else:
                display(pd.DataFrame(rows))
    # Xem tối đa một hình mỗi loại; mọi ảnh khác vẫn được lưu trong examples/.
    from PIL import Image as PILImage
    shown = set()
    for example in globals().get("FINAL_TEST_EXAMPLES", []):
        if example["category"] not in shown:
            print(example["category"], example["sample_id"])
            with PILImage.open(FINAL_TEST_DIR / "examples" / example["figure"]) as source:
                display(source.copy())
            shown.add(example["category"])
else:
    print("Chưa có test hoàn chỉnh. Không dùng bảng validation thay cho bảng test.")


#### Cách đọc kết quả hiện tại

Dùng bảng test vừa xuất ở trên, đúng validation_lock_sha256, thiết bị và phiên bản tương ứng. So hybrid với cả full640 và full khớp pixel; đọc F1/accuracy cùng coverage, FVR, MAE và tổng thời gian. Không chọn lại tham số từ test. Validation có thiên lệch lựa chọn; không dùng số validation thay test.

> **Lịch sử cấu hình cũ:** phần nhận xét lưu dưới đây thuộc lượt notebook cũ, chưa được xác nhận là kết quả của cấu hình khóa mới. Giữ để truy vết, không chép các con số này vào bảng/kết luận của lượt mới. Mọi kết luận mới phải lấy từ artifact mới ở mục 6.3.

<details>
<summary>Nhận xét lịch sử của lượt cũ — không phải bảng test mới</summary>

#### Cách đọc và thảo luận kết quả

**Nhận xét theo số liệu test (within_camera, 270 ảnh).** Các số dưới đây lấy từ lần chạy hiện tại; chạy lại có thể lệch nhẹ, cần đối chiếu với bảng ở trên.

- **HOG + SVM đạt F1 cao nhất (0.976) và nhanh nhất (283 ms/ảnh)**, tỉ lệ báo trống sai thấp nhất (0.018). Lưu ý: HOG được huấn luyện trên chính ba camera, còn YOLO11n chưa fine-tune, nên đây không phải so sánh cùng điều kiện học.
- **YOLO full-frame rất phụ thuộc camera và thời tiết**: F1 0.42 ở PUCPR/rainy và 0.40 ở PUCPR/sunny, nhưng 0.97–1.00 ở UFPR04/UFPR05. Hai nhóm PUCPR này có rất nhiều xe (trung bình 77 và 42 ô có xe/ảnh trên 100 ô), xe nhỏ và sát nhau; recall toàn cục của full-frame chỉ 0.45 và tỉ lệ báo trống sai 0.51. Nguyên nhân là giả thuyết phù hợp với số đo (xe nhỏ, dày đặc, bị che), chưa được kiểm chứng riêng.
- **ROI/tile cải thiện rõ ở PUCPR có nhiều xe** (rainy 0.42 → 0.83, sunny 0.40 → 0.80) nhưng không tốt hơn full-frame ở UFPR (UFPR04 0.954 so với 0.970, UFPR05 0.982 so với 0.990). ΔF1 gộp +0.243 vì thế chủ yếu đến từ PUCPR. Cái giá là thời gian ×3.4 và 3.3 lượt suy luận/ảnh.
- **Các nhóm rất ít ô có xe không dùng để so sánh F1.** Trong test, PUCPR/overcast chỉ có 6 ô có xe (trên 3.000 ô) và UFPR04/overcast chỉ có 1 ô có xe (trên 840 ô). F1 của hai nhóm này (ví dụ ROI/tile ở UFPR04/overcast bằng 0.00 vì bỏ sót đúng ô có xe duy nhất, còn full-frame bằng 1.00) phản ánh một vài ô chứ không phải chất lượng thật; accuracy của các nhóm này vẫn cao (≥0.83).
- **Chính sách hình học yếu** giảm tỉ lệ báo trống sai nhưng làm coverage rơi xuống khoảng 0.92–0.93; mức giảm lỗi nhỏ (ví dụ HOG 0.0214 → 0.0180) so với số ô chuyển thành unknown (hơn 1.000 ô/phương pháp). Với bộ dữ liệu này, cờ hình học là một công cụ cảnh báo hơn là cải thiện độ chính xác.

**Cách đọc các chỉ số.**

- **Không so sánh bằng accuracy covered-only.** Phương pháp báo unknown nhiều có thể đạt accuracy cao trên các ô đã xác định, nên phải đọc F1/accuracy tổng cùng với coverage.
- **Tỉ lệ báo trống sai** (`false_vacancy_rate`) quan trọng khi ứng dụng: báo trống một ô có xe sẽ đưa người dùng tới chỗ không còn trống. Chính sách hình học yếu nhằm giảm lỗi này, đổi lại coverage giảm. Mức đánh đổi xem ở bảng `weak_geometry_policy`.
- **ROI/tile so với full-frame**: tile tăng độ phân giải hiệu dụng cho xe nhỏ ở xa nhưng cần nhiều lượt suy luận. Kết luận dựa trên ΔF1/ΔMAE và tỉ lệ thời gian đo được.
- **HOG/SVM** học trên chính các camera của giao thức `within_camera`, còn YOLO11n là checkpoint COCO pretrained chưa fine-tune. Khác biệt dữ liệu học này cần được nêu khi so sánh.
- **Theo camera/thời tiết**: mỗi nhóm validation/test có 30 ảnh, nên chênh lệch nhỏ có thể chỉ là dao động.
- **Giao thức chuyển camera** chưa được chạy trong lần nộp này (mọi kết quả ở trên là within_camera). Để chạy: đặt `PROTOCOL = "transfer_ufpr04_to_ufpr05"` ở mục 2 và chạy lại toàn bộ. Layout UFPR05 không có ảnh train nên không có tham chiếu hình học; cờ hình học không bật và điều này được ghi rõ trong đầu ra.

</details>


## 8. Kết luận


Notebook tích hợp đầy đủ luồng dữ liệu → phương pháp → evaluator → validation/khóa → test → bảng và ảnh trong cùng runtime. Có ba phương pháp chính (HOG/SVM, YOLO toàn ảnh, hybrid) và đối chứng YOLO toàn ảnh khớp pixel. Vùng ô là cấu hình đã biết; Canny/Hough kiểm tra hình học, không tự sinh trạng thái trống.

**Kết luận thực nghiệm mới phải dựa trên artifact đúng khóa ở mục 6.3.** Khi chưa chạy test mới, chỉ báo kết quả validation và các giới hạn lựa chọn tham số. Hybrid cần đối chiếu full640 lẫn full khớp pixel cùng thời gian/MAE, không tự nhận cải tiến.

Giới hạn còn lại: camera cố định, nhãn vùng ô có sẵn, detector pretrained chưa fine-tune, tập con theo quota và ít camera/ngày. Split chuyển camera đã chuẩn bị nhưng không có nghĩa đã thực hiện lượt đánh giá đó. Lịch sử notebook từng có output test cần được công bố.

<details>
<summary>Nội dung kết luận của lượt cũ — lưu lịch sử, không phải kết quả cấu hình mới</summary>

Notebook hiện thực trọn chuỗi từ ảnh toàn cảnh đến trạng thái từng ô và số chỗ trống:

1. PKLot được kiểm tra và chia theo ngày bằng manifest khóa `pklot_v1`, dùng chung cho mọi phương pháp.
2. Ba phương pháp chạy trong cùng một pipeline: HOG/SVM trên từng ô, YOLO11n full-frame, YOLO11n trên ROI/tile với quy đổi box và global NMS. Hai phương pháp detector dùng chung một quy tắc ghép xe–ô.
3. Canny/Hough kiểm tra hình học. So với tham chiếu train, pipeline phát hiện camera lệch, ảnh suy giảm và ô mất vạch. Khi hình học yếu, hệ thống thông báo và trả **unknown** thay vì tự coi ô là trống.
4. Mỗi ảnh được trực quan hóa với 6 panel và xuất CSV/JSON. Đánh giá dùng bộ đánh giá chung, có tính unknown, MAE đếm và thời gian trên ảnh gốc.

**Kết quả chính (within_camera, test).** HOG + SVM cho F1 cao nhất (0.976) và nhanh nhất (283 ms/ảnh), nhưng được học trên PKLot. Trong hai phương pháp dùng YOLO11n pretrained, ROI/tile hơn full-frame +0.243 F1 và giảm MAE số chỗ trống từ 12.6 xuống 7.0, với thời gian ×3.4. Phần lớn lợi ích đến từ PUCPR (xe nhỏ, ở xa); ở UFPR04/UFPR05, full-frame đã đạt F1 0.97–0.99 và tile không cải thiện thêm. Chi tiết và các giới hạn của so sánh nằm ở mục 7.

**Giới hạn.**
- YOLO11n chưa fine-tune trên PKLot. `imgsz` được chọn trên validation (4.4) nhưng xe nhỏ ở xa vẫn có thể bị bỏ sót.
- Ngưỡng Canny và ngưỡng chất lượng hình học là giá trị mặc định, chưa tối ưu theo từng camera.
- Hệ thống giả định polygon ô có sẵn và camera cố định. Cờ lệch camera chỉ phát hiện, không tự hiệu chỉnh layout.
- Tập con cân bằng thời tiết nên không đại diện tỉ lệ tự nhiên của PKLot.
- Chỉ chạy giao thức within_camera; kiểm thử chuyển camera (UFPR04 → UFPR05) chưa được thực hiện.
- Mỗi nhóm camera/thời tiết chỉ có 30 ảnh test, nên chênh lệch nhỏ giữa các nhóm chưa đủ để kết luận.

**Hướng phát triển.**
- Fine-tune detector trên ảnh bãi xe.
- Hiệu chỉnh ngưỡng hình học theo camera.
- Tự căn lại layout theo độ lệch đo được.
- Kết hợp điểm HOG với detector.
- Thử thêm CNRPark-EXT.

</details>


## 9. Hướng dẫn tái lập


**Một notebook từ chuẩn bị tới kết quả:**

1. Mục 2: chọn local/Drive, đường dẫn dataset, REF hoặc bundle và các cờ chạy.
2. Mục 3: dùng lại dataset/audit và manifest khóa. Có file local thì không tải lại code/split/plan.
3. Mục 4–5: các demo tùy chọn, mặc định bỏ vì runner chung làm các bước này.
4. Mục 6.1: self-check evaluator.
5. Mục 6.2: có khóa rồi giữ RUN_VALIDATION_SEARCH=False; cell chỉ đọc lại artifact. Chưa có thì chủ động bật tìm kiếm trước khi test.
6. Mục 6.3a: kiểm tra đầy đủ khóa/model/phiên bản và dataset. Mục 6.3b: bật RUN_FINAL_TEST để chạy test thật. Giữ False để đọc kết quả đã có.
7. Mục 6.3c–e: bảng, ảnh đúng/sai/unknown và checksum từ dự đoán lưu; mục 7 hiện bảng/ảnh trên notebook.
8. Giữ toàn bộ output bền vững/Drive, không chỉ screenshot. Không cần notebook 02; file đó chỉ còn là bản thực hành tùy chọn.

Không có chế độ rút gọn để nhận là test chính thức. Muốn smoke-test riêng phải lưu riêng và không đưa vào bảng cuối. Không ghi đè output test đã có; chọn tên run mới và công bố lịch sử khi chạy lại do lỗi kỹ thuật.

Giữ checkpoint, model HOG, tham chiếu, manifests/layout, plan/khóa và code/môi trường. Gói ZIP dùng được cho runtime mới nhưng không chứa ảnh dataset và không bảo đảm mọi phiên bản thư viện có wheel trên mọi Colab.


In [ ]:
print("Nguồn engine:", globals().get("provenance"))
if globals().get("TASK12_DIR"):
    print("Validation artifact:", TASK12_DIR)
    print("Khóa:", TASK12_DIR / "locked_config.json")
if globals().get("FINAL_TEST_DIR"):
    print("Thư mục test:", FINAL_TEST_DIR)
    print("Bảng:", FINAL_TEST_DIR / "tables")
    print("Ảnh và quy tắc chọn:", FINAL_TEST_DIR / "examples")
    print("Hash bằng chứng:", FINAL_TEST_DIR / "evidence_sha256.json")
globals().get("sync_outputs_to_drive", lambda: None)()
